# PolEmo2 — Pierwsze modele klasyfikacji tekstu

W kolejnych komórkach znajdziesz objaśnienia, kod i wyniki doświadczeń.
Czytaj je po kolei i uruchamiaj kod przyciskiem ▶ albo **Shift+Enter**.
Najpierw wykonamy krótkie przykłady. Późniejsze definicje funkcji
przygotują właściwe doświadczenie, które uruchomimy w komórce
**Uruchomienie porównania**.

Uczymy model rozpoznawać wydźwięk polskich opinii. W jednym doświadczeniu
używamy całego zbioru. W drugim uczymy model na tekstach o hotelach
i testujemy go na tekstach medycznych. Każdy model dopasowujemy osobno.

Wybierz **Plik → Zapisz kopię na Dysku**.
W tej części wystarczy zwykły procesor (CPU).
Materiały, dane i obliczenia nie wymagają płatnych usług ani klucza API.

[Artykuł i materiały](https://neuronovo.si/projekty/polemo2/).


In [ ]:
import importlib.util
import subprocess
import sys

# Colab udostępnia większość bibliotek. Instalujemy tylko brakujące,
# z oficjalnego rejestru PyPI; nie zmieniamy całego środowiska.
pakiety = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib'}
brakujace = [nazwa_pypi for modul, nazwa_pypi in pakiety.items()
             if importlib.util.find_spec(modul) is None]
if brakujace:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *brakujace])
print("Biblioteki są dostępne.")


## Zanim zaczniemy: jak pracować z tym materiałem

Ten materiał jest przeznaczony także dla osób z pierwszego semestru studiów. Zaczniemy od kilku poleceń Pythona i małego przykładu. Znajomość sieci neuronowych, rachunku prawdopodobieństwa ani pochodnych nie jest potrzebna do wykonania pierwszej części.

Na pierwszych zajęciach przejdź przez przykłady do nagłówka **Pierwszy klasyfikator**. Sprawdź, czy umiesz wyjaśnić jego dane wejściowe i wynik. Następnie poznaj podział danych i przejdź do pełnego doświadczenia. Dostrajanie modelu językowego możesz zostawić na kolejne zajęcia.

Komórka tekstowa zawiera objaśnienia. Komórka kodu zawiera polecenia, które wykona Python. **Shift+Enter** uruchamia bieżącą komórkę i przechodzi do następnej. Wynik pojawia się pod kodem. Kolejność ma znaczenie: nazwa utworzona w jednej komórce może być używana w kolejnych.

Liczba w nawiasach przy komórce oznacza kolejność jej wykonania w danej sesji. Po ponownym uruchomieniu środowiska wykonaj komórki od góry, żeby odtworzyć zmienne i funkcje.


## Kilka poleceń Pythona, które będą nam potrzebne

**Zmienna** to nazwa, pod którą przechowujemy wartość. Tekst zapisany w cudzysłowie jest napisem (ang. string). Znak `=` przypisuje wartość do nazwy. Funkcja `print(...)` wyświetla wynik. Wiersz zaczynający się od `#` jest komentarzem dla czytelnika; Python go pomija.


In [ ]:
# Cudzysłowy oznaczają, że przechowujemy tekst.
# Nazwa po lewej stronie znaku = pozwoli później odwołać się do tej wartości.
opinia = "Obsługa była miła i pomocna."

# Etykieta to kategoria przypisana do przykładu.
# W tym projekcie nazwa plus oznacza wydźwięk pozytywny.
etykieta = "plus"

# W nawiasach podajemy wartości, które chcemy wyświetlić.
print(opinia)
print("Przypisana kategoria:", etykieta)


### Lista, pozycja i pętla

**Lista** przechowuje kilka wartości w ustalonej kolejności. Tworzymy ją w nawiasach kwadratowych. Numerowanie pozycji zaczyna się od zera, więc `opinie_pokazowe[0]` to pierwszy tekst. `len(...)` zwraca liczbę elementów listy.

**Pętla** `for` wykonuje ten sam fragment kodu kolejno dla wielu elementów. Wcięcie pod wierszem kończącym się dwukropkiem pokazuje, które polecenia należą do pętli. Zachowaj je podczas edycji.


In [ ]:
# Każdy element listy jest osobnym tekstem. Przecinki rozdzielają elementy.
opinie_pokazowe = [
    "Miła obsługa i czysty pokój.",
    "Brudny pokój i niemiła obsługa.",
    "Hotel ma cztery piętra.",
]

print("Liczba opinii:", len(opinie_pokazowe))
print("Pierwsza opinia:", opinie_pokazowe[0])

# Przy każdym wykonaniu pętli zmienna tekst otrzyma kolejny element listy.
for tekst in opinie_pokazowe:
    print(tekst)


### Słownik i funkcja

**Słownik** łączy klucze z wartościami. Pozwala np. znaleźć polskie objaśnienie etykiety. W zapisie `opisy["minus"]` kluczem jest napis `minus`.

**Funkcja** to nazwany fragment kodu, który możemy wykonywać wielokrotnie. `def` rozpoczyna jej definicję. Nazwa w nawiasie definicji, tutaj `tekst`, jest parametrem funkcji. Przy wywołaniu przekazujemy argument, np. konkretną opinię. `return` oddaje wynik do dalszych obliczeń. Samo zdefiniowanie funkcji jeszcze jej nie uruchamia.


In [ ]:
# Nawiasy klamrowe tworzą słownik. Dwukropek oddziela klucz od wartości.
opisy_etykiet = {
    "zero": "neutralna",
    "minus": "negatywna",
    "plus": "pozytywna",
    "amb": "mieszana",
}
print("Znaczenie etykiety minus:", opisy_etykiet["minus"])

# Parametr tekst otrzyma wartość podaną przy wywołaniu funkcji.
def policz_slowa_pokazowo(tekst):
    # split() dzieli napis w miejscach białych znaków, np. spacji.
    slowa = tekst.split()
    # Wynikiem funkcji jest liczba elementów powstałej listy.
    return len(slowa)

# Dopiero ten wiersz wywołuje funkcję. Wynik przypisujemy do zmiennej.
liczba_slow = policz_slowa_pokazowo(opinie_pokazowe[0])
print("Liczba słów w pierwszej opinii:", liczba_slow)


## Jak tekst staje się zbiorem cech

**Cecha** to liczba opisująca jakiś aspekt przykładu. W najprostszym zapisie jedna cecha odpowiada liczbie wystąpień określonego słowa. Słownik określa, którym słowom przypisujemy osobne kolumny. Wiersz opisujący tekst nazywamy **wektorem cech**, a tabelę takich wierszy — **macierzą cech**.

Dla trzech tekstów otrzymamy następujące liczby wystąpień:

| Tekst | dobry | hotel | pokój | zły |
|---|---:|---:|---:|---:|
| dobry pokój | 1 | 0 | 1 | 0 |
| dobry hotel | 1 | 1 | 0 | 0 |
| zły pokój | 0 | 0 | 1 | 1 |

Ten zapis nie zachowuje kolejności słów i bywa nazywany workiem słów (ang. bag of words). W dużym słowniku pojedynczy tekst wykorzystuje tylko niewielką część kolumn. **Macierz rzadka** przechowuje przede wszystkim wartości niezerowe, dzięki czemu zajmuje mniej pamięci.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
from IPython.display import display

# Używamy trzech krótkich, wymyślonych tekstów.
teksty_do_liczenia = ["dobry pokój", "dobry hotel", "zły pokój"]

# CountVectorizer zbuduje słownik i policzy wystąpienia słów.
licznik_slow = CountVectorizer()
macierz_liczebnosci = licznik_slow.fit_transform(teksty_do_liczenia)
nazwy_cech_pokazowe = licznik_slow.get_feature_names_out()

# Tylko w tym małym przykładzie zamieniamy macierz rzadką na zwykłą tablicę.
# Właściwe dane PolEmo2 pozostawiamy w postaci rzadkiej, aby oszczędzić pamięć.
tabela_liczebnosci = pd.DataFrame(
    macierz_liczebnosci.toarray(),
    columns=nazwy_cech_pokazowe,
)
display(tabela_liczebnosci)


### Co zmienia TF-IDF

TF-IDF nadaje cechom wagi. **TF** (ang. term frequency) opisuje występowanie słowa w danym tekście. **IDF** (ang. inverse document frequency) uwzględnia, w jak wielu tekstach słowo występuje. Słowo pojawiające się w prawie każdym dokumencie otrzymuje mniejszą wagę IDF niż słowo spotykane w niewielu dokumentach.

W naszym przykładzie `dobry` występuje w dwóch tekstach, a `hotel` w jednym, dlatego `hotel` ma wyższe IDF. Wagi nie mówią jeszcze, czy słowo jest pozytywne lub negatywne. Tego powiązania uczy się dopiero klasyfikator na podstawie etykiet.

Scikit-learn domyślnie stosuje wygładzanie IDF i normalizację długości wektora. W efekcie ostateczna wartość zależy także od pozostałych słów tekstu. W badaniu ustawiamy dodatkowo `sublinear_tf=True`: wraz z kolejnymi powtórzeniami tego samego słowa jego wkład rośnie wolniej. Słownik i IDF wyznaczamy wyłącznie na treningu.


In [ ]:
# fit_transform oznacza: dopasuj słownik oraz IDF, a potem przekształć teksty.
# Dopasowanie dotyczy tutaj tylko trzech tekstów demonstracyjnych.
tfidf_do_pokazu = TfidfVectorizer()
macierz_tfidf_pokazowa = tfidf_do_pokazu.fit_transform(teksty_do_liczenia)

# Nazwy kolumn opisują cechy; w wierszach są kolejne teksty.
tabela_tfidf_pokazowa = pd.DataFrame(
    macierz_tfidf_pokazowa.toarray(),
    columns=tfidf_do_pokazu.get_feature_names_out(),
)
# Zaokrąglamy jedynie widok tabeli. Wartości używane w obliczeniach nie zmieniają się.
display(tabela_tfidf_pokazowa.round(3))

# zip łączy nazwę cechy z odpowiadającą jej wagą IDF.
for slowo, waga_idf in zip(tfidf_do_pokazu.get_feature_names_out(), tfidf_do_pokazu.idf_):
    print(slowo, "→ IDF =", round(waga_idf, 3))

# Nowy tekst przekształcamy tym samym słownikiem i tymi samymi wagami.
# transform nie uczy ponownie słownika ani IDF.
nowy_wektor_pokazowy = tfidf_do_pokazu.transform(["dobry pokój hotelowy"])
print("Liczba wierszy i kolumn nowej macierzy:", nowy_wektor_pokazowy.shape)


### Dlaczego uwzględniamy pary słów

`polecam` i `nie polecam` mogą wyrażać przeciwną ocenę. **Bigram** to para kolejnych słów, np. `nie polecam`. Ustawienie `ngram_range=(1, 2)` uwzględnia zarówno pojedyncze słowa, jak i ich pary. Przy ciągach znaków rozpatrujemy fragmenty zapisu, co może pomóc przy odmianie wyrazów.

Naiwny klasyfikator Bayesa korzysta z informacji o cechach występujących w poszczególnych klasach. Przy łączeniu tych informacji przyjmuje upraszczające założenie: cechy są niezależne, jeśli znamy klasę tekstu. Hiperparametr `alpha` odpowiada za wygładzanie i zapobiega m.in. problemom z cechą niewidzianą wcześniej w danej klasie.

Regresja logistyczna wyznacza wagi wskazujące, jak cechy przemawiają za poszczególnymi klasami. Z uzyskanych wyników oblicza prawdopodobieństwa klas. W zadaniu wybiera jedną z czterech etykiet. Hiperparametr `C` steruje siłą regularyzacji, czyli ograniczenia wag podczas uczenia: mniejsze `C` oznacza silniejsze ograniczenie.

**Pytania:** czy wektor nowego tekstu powinien mieć inny słownik niż wektory uczące? Co stanie się ze słowem `hotelowy`, jeśli nie ma go w słowniku z naszego małego przykładu?


## Pierwszy klasyfikator na wymyślonych opiniach

**Klasyfikacja** polega na przypisaniu jednej z ustalonych kategorii. Przykładem jest rozpoznanie, czy wydźwięk opinii jest pozytywny, negatywny, neutralny czy mieszany. **Przykład uczący** składa się z tekstu i jego prawidłowej etykiety. **Przewidywanie** to etykieta wskazana przez model dla podanego tekstu.

Poniższe zdania są wymyślone na potrzeby objaśnienia kodu. Uczymy na nich mały model i oglądamy kilka przewidywań. Ta część pokazuje działanie poleceń. Rzetelną ocenę jakości przeprowadzimy później na oddzielnych częściach zbioru PolEmo2.

Biblioteka scikit-learn udostępnia gotowe algorytmy. `import` pozwala z nich skorzystać. `TfidfVectorizer` zamieni tekst na cechy liczbowe, a `LogisticRegression` nauczy się wybierać kategorię na podstawie tych cech. W poniższym potoku połączymy oba elementy.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Pierwsza lista zawiera teksty. Druga zawiera ich prawidłowe etykiety.
# Elementy na tych samych pozycjach tworzą parę: tekst i jego etykieta.
teksty_do_pokazu = [
    "Miła obsługa i bardzo dobry pobyt.",
    "Czysty pokój, polecam ten hotel.",
    "Wspaniała obsługa, jestem zadowolona.",
    "Brudny pokój, nie polecam.",
    "Niemiła obsługa i bardzo zły pobyt.",
    "Fatalny hotel, jestem niezadowolona.",
    "Hotel znajduje się przy ulicy Leśnej.",
    "Budynek hotelu ma cztery piętra.",
    "W hotelu jest dwadzieścia pokoi.",
    "Miła obsługa, ale brudny pokój.",
    "Czysty pokój, ale niemiła obsługa.",
    "Dobry hotel, ale pobyt był za drogi.",
]
etykiety_do_pokazu = [
    "plus", "plus", "plus",
    "minus", "minus", "minus",
    "zero", "zero", "zero",
    "amb", "amb", "amb",
]

# Potok (ang. pipeline) wykona kolejno dwa kroki:
# 1. TF-IDF opisze tekst liczbami. Uwzględni słowa i pary kolejnych słów.
# 2. Regresja logistyczna będzie klasyfikatorem tych liczbowych opisów.
model_pokazowy = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression(C=1.0, max_iter=1000),
)

# fit oznacza dopasowanie do przykładów uczących.
# Pierwszy argument to teksty, a drugi to prawidłowe odpowiedzi dla tych tekstów.
model_pokazowy.fit(teksty_do_pokazu, etykiety_do_pokazu)

# Tych tekstów nie podawaliśmy wcześniej w identycznej postaci.
nowe_teksty = [
    "Bardzo miła obsługa, polecam.",
    "Brudny pokój i fatalny pobyt.",
    "Czysty pokój, ale zły pobyt.",
]

# predict korzysta z dopasowanego już modelu i zwraca tablicę przewidywań.
przewidywania_pokazowe = model_pokazowy.predict(nowe_teksty)

# zip łączy teksty z odpowiadającymi im przewidywaniami według pozycji.
# W każdej iteracji odbieramy dwie wartości: tekst oraz przewidzianą etykietę.
for tekst, przewidywana_etykieta in zip(nowe_teksty, przewidywania_pokazowe):
    print(przewidywana_etykieta, "→", tekst)


### Zmień klasyfikator i porównaj przewidywania

Zachowamy te same teksty, etykiety i ustawienia TF-IDF. Zamiast regresji logistycznej użyjemy naiwnego klasyfikatora Bayesa. Tabela pokaże odpowiedzi obu modeli na tych samych trzech nowych tekstach.

Nie obliczamy tutaj oceny jakości. Kilkanaście wymyślonych zdań służy do poznania kodu; pełne porównanie przeprowadzimy z użyciem treningu, walidacji i testu PolEmo2.


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# W potoku zmieniamy drugi krok, czyli algorytm przypisujący klasę.
# Reprezentacja tekstu i przykłady uczące pozostają takie same.
model_bayesa_pokazowy = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    MultinomialNB(alpha=1.0),
)
model_bayesa_pokazowy.fit(teksty_do_pokazu, etykiety_do_pokazu)
odpowiedzi_bayesa_pokazowe = model_bayesa_pokazowy.predict(nowe_teksty)

# Każdy wiersz tabeli opisuje ten sam tekst i dwie przewidziane etykiety.
porownanie_pokazowe = pd.DataFrame({
    "tekst": nowe_teksty,
    "regresja logistyczna": przewidywania_pokazowe,
    "naiwny Bayes": odpowiedzi_bayesa_pokazowe,
})
display(porownanie_pokazowe)


### Zatrzymaj się i sprawdź

1. Które dane przekazaliśmy do `fit`, a które do `predict`?
2. Jak zmienić tekst nowej opinii, żeby poprosić model o kolejne przewidywanie?
3. Dlaczego ocena zaledwie trzech przykładów nie wystarcza do oceny jakości modelu?
4. Co może utrudniać rozpoznanie tekstu zawierającego jednocześnie pochwałę i skargę?

Możesz dopisać własną wymyśloną opinię do `nowe_teksty`. W tej części sprawdzamy działanie poleceń na przykładach demonstracyjnych; nie modyfikujemy danych ani wyników badania PolEmo2.


## Od przykładu do doświadczenia

W badaniu potrzebujemy znacznie więcej przykładów oraz ustalonej metody oceny. **Zbiór uczący** służy do dopasowania modelu. **Zbiór walidacyjny** służy do wyboru ustawień, np. siły regularyzacji lub liczby epok. **Zbiór testowy** służy do końcowej oceny wybranego rozwiązania.

**Parametry modelu** to liczby wyznaczane podczas uczenia, np. wagi cech. **Hiperparametry** to ustawienia, które wybieramy przed danym uczeniem, np. `C`, `alpha` lub tempo uczenia. W naszym porównaniu sprawdzamy kilka ustalonych wartości i wybieramy je według walidacji.

**Przeuczenie** (ang. overfitting) występuje, gdy model dobrze dopasowuje się do przykładów uczących, lecz gorzej radzi sobie z nowymi. **Generalizacja** oznacza zdolność działania na nowych przykładach. W próbie hotele → medycyna sprawdzamy ją po zmianie tematyki tekstów.

Poniżej przygotujemy wspólny kod całego doświadczenia. Każda funkcja ma objaśnienia i komentarze. Nie musisz od razu samodzielnie odtwarzać pobierania plików, pomiaru czasu ani zapisu raportów. Zwróć najpierw uwagę na dane, przekształcenia tekstu, `fit`, wybór na walidacji i `predict`. Funkcje z nazwą zaczynającą się od `_` są funkcjami pomocniczymi; podkreślenie jest tu konwencją nazewniczą Pythona.


### Jak czytać dalszy kod

| Zapis | Znaczenie w tym notatniku |
|---|---|
| `import pandas as pd` | Wczytuje bibliotekę pandas; skrót `pd` będzie jej nazwą w kodzie. |
| `obiekt.metoda(...)` | Wywołuje operację udostępnioną przez dany obiekt. |
| `tabela["target"]` | Pobiera kolumnę o nazwie `target` z tabeli pandas. |
| `if warunek:` | Wykonuje wcięty blok, jeśli warunek jest spełniony. |
| `==` | Sprawdza równość; pojedyncze `=` służy do przypisania. |
| `None` | Oznacza brak wartości, np. brak wczytanego jeszcze testu. |
| `True`, `False` | Wartości logiczne: prawda i fałsz. |
| `a, b = funkcja()` | Odbiera dwa wyniki zwrócone przez funkcję. |
| `[f(x) for x in lista]` | Tworzy listę wyników przekształcenia kolejnych elementów. |
| `lambda x: ...` | Krótki zapis funkcji, np. wskazującej, według czego porównać wiersze. |

`context` jest obiektem grupującym dane i ustawienia jednego doświadczenia. Przykładowo `context.train` zawiera tabelę uczącą, a `context.validation` — walidacyjną. `row` jest słownikiem, do którego dopisujemy wynik jednej metody. Dzięki temu wyniki wszystkich metod mają taki sam układ.


## Biblioteki, źródła i nazwy metod

Poniżej widać importy i stałe: wersję danych, wersję modelu, źródło
polskich wektorów fastText oraz podziały obu doświadczeń. Nie pobieramy
najnowszej, zmieniającej się wersji modelu: wskazujemy konkretną rewizję.
`ORIGINAL_MODULE_SHA256` dokumentuje źródło tej implementacji.
Pole `module_sha256` w raporcie odnosi się do tego oryginalnego pliku,
z którego funkcje rozdzielono na komórki; nie jest sumą całego notatnika.


In [ ]:
# To ustawienie dotyczy sposobu zapisywania informacji o typach argumentów i wyników funkcji w
# Pythonie. Pozostawiamy je zgodnie z kodem źródłowym doświadczenia.
from __future__ import annotations
# Instrukcja import udostępnia kod z biblioteki. Biblioteki poniżej pomagają kopiować dane,
# obsługiwać pliki, pobierać zasoby i mierzyć czas; właściwe metody uczenia pojawią się dalej.
import copy
import gc
# hashlib oblicza sumy kontrolne plików. Zapisujemy je, żeby można było później sprawdzić, z
# jakich dokładnie danych korzystało doświadczenie.
import hashlib
import importlib.metadata
# JSON to tekstowy format zapisu list i par klucz–wartość. Użyjemy go do zapisania konfiguracji
# oraz wyników.
import json
import os
import platform
# Ten moduł dostarcza generator liczb pseudolosowych. Kontrolowana losowość przyda się przy
# przygotowaniu uczenia sieci.
import random
# Moduł re wyszukuje fragmenty tekstu zgodne ze wzorcem, czyli wyrażeniem regularnym. Posłuży do
# odczytania etykiet i podziału tekstu na słowa.
import re
import shutil
import sys
import time
import urllib.request
from datetime import datetime, timezone
# Zapis from ... import ... udostępnia wybraną nazwę z biblioteki. Path pozwala wygodnie składać
# ścieżki do plików i tworzyć katalogi.
from pathlib import Path
# NumPy służy do obliczeń na tablicach liczb. Zapis as np nadaje bibliotece krótszą nazwę, dlatego
# dalej zobaczysz na przykład np.zeros i np.mean.
import numpy as np
# pandas obsługuje dane w tabelach. DataFrame będzie tabelą, w której jeden wiersz odpowiada
# jednej opinii, a kolumny zawierają tekst i jego etykietę.
import pandas as pd
# scikit-learn udostępnia gotowe narzędzia uczenia maszynowego. DummyClassifier da prosty punkt
# odniesienia: zawsze wskaże klasę najczęstszą w treningu.
from sklearn.dummy import DummyClassifier
# Klasyfikator potrzebuje liczb. TfidfVectorizer przygotuje je na podstawie słów i ich częstości
# występowania w tekstach.
from sklearn.feature_extraction.text import TfidfVectorizer
# Regresja logistyczna, mimo słowa regresja w nazwie, posłuży tutaj do wyboru jednej z czterech
# klas wydźwięku.
from sklearn.linear_model import LogisticRegression
# Te funkcje porównują odpowiedzi modelu z prawidłowymi etykietami. Każda opisuje jakość
# klasyfikacji w trochę inny sposób.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
# MultinomialNB to naiwny klasyfikator Bayesa. W tym doświadczeniu otrzyma nieujemne cechy TF-IDF.
from sklearn.naive_bayes import MultinomialNB
# FeatureUnion połączy dwa zestawy cech, a make_pipeline połączy kolejne kroki przetwarzania, na
# przykład skalowanie i klasyfikację.
from sklearn.pipeline import FeatureUnion, make_pipeline
# StandardScaler przelicza każdą cechę względem jej średniej i skali zmienności w treningu.
# Użyjemy go dla gęstych wektorów fastText i enkodera.
from sklearn.preprocessing import StandardScaler
# SimpleNamespace zbiera powiązane wartości w jednym obiekcie. Zamiast wielu osobnych argumentów
# będziemy przekazywać context, a jego elementy odczytywać na przykład jako context.train.
from types import SimpleNamespace

# To identyfikator pliku, z którego pochodzi sprawdzona implementacja. Raport zachowa ten
# identyfikator, choć w notatniku kod podzielono na osobne komórki.
ORIGINAL_MODULE_SHA256 = "44b4be5cbf87b2d85696d5e6c7bbb6109ced6816a687695c85296be7376ef015"

# Ustalamy ziarno generatorów losowych. Przekazanie tej samej wartości bibliotekom pomaga
# odtworzyć losowanie kolejności przykładów i początkowych wag; sama liczba 42 nie ma szczególnego
# znaczenia dla zadania.
SEED = 42

# Stałe zapisane wielkimi literami przechowują wspólne ustawienia. Identyfikator i rewizja danych
# wskazują konkretny zbiór oraz jego wersję.
DATASET_ID = "clarin-pl/polemo2-official"

DATASET_REVISION = "802e35d2b12bae84bb07911d841e8f046dc2fcef"

# Enkoder (ang. encoder) to model, który przetwarza tekst na reprezentacje liczbowe. Zapisujemy
# nazwę i dokładną wersję polskiego modelu, aby oba doświadczenia zaczynały od tych samych wag.
ENCODER_ID = "sdadas/polish-distilroberta"

ENCODER_REVISION = "849b664fa3134beae84095d28a184c145c6a3aa5"

# Pod tym adresem znajduje się oficjalny plik gotowych polskich wektorów fastText. Jego pobranie
# nie wymaga płatnego API.
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pl.vec"

# Nawiasy kwadratowe tworzą listę. Pozycje liczymy od zera: 0 oznacza zero, 1 minus, 2 plus, a 3
# amb. Klasa amb obejmuje opinie zawierające elementy pozytywne i negatywne.
LABELS = ["zero", "minus", "plus", "amb"]

# Nawiasy klamrowe zapisują tu słownik: identyfikator metody jest kluczem, a czytelna polska nazwa
# jego wartością. Identyfikator służy kodowi, nazwa trafi do tabeli wyników.
METHOD_NAMES = {
    "dummy": "Najczęstsza klasa",
    "tfidf_nb": "TF-IDF słów + naiwny klasyfikator Bayesa",
    "tfidf_lr": "TF-IDF słów + regresja logistyczna",
    "tfidf_word_char_lr": "TF-IDF słów i fragmentów słów + regresja logistyczna",
    "fasttext_lr": "Średnia wektorów fastText PL + regresja logistyczna",
    "encoder_frozen_lr": "Zamrożony polski encoder + regresja logistyczna",
    "encoder_finetuned": "Dostrajanie polskiego encodera",
}

# list(METHOD_NAMES) pobiera klucze słownika w kolejności ich zapisania. Fragment [:5] wybiera
# pięć pierwszych pozycji, o indeksach od 0 do 4.
CLASSICAL_METHODS = list(METHOD_NAMES)[:5]

# Fragment [5:] wybiera pozycje od indeksu 5 do końca listy. Są to dwie metody korzystające z
# enkodera.
ENCODER_METHODS = list(METHOD_NAMES)[5:]

# Ten zagnieżdżony słownik opisuje dwa doświadczenia i ich podziały danych. Para taka jak
# ('hotels', 'dev') jest krotką: przechowuje kolejno domenę tekstów i nazwę podziału. Zachowujemy
# oficjalne podziały autora.
PROTOCOLS = {
    "all_text": {
        "train": ("all", "train"),
        "validation": ("all", "dev"),
        "test": ("all", "test"),
    },
    "hotels_to_medicine": {
        "train": ("hotels", "train"),
        "validation": ("hotels", "dev"),
        "test": ("medicine", "test"),
    },
}


## Pobieranie plików i sumy kontrolne

Dane pobieramy porcjami na dysk. Cały duży plik fastText nie musi
jednocześnie zajmować pamięci RAM. Suma SHA256 pozwala sprawdzić,
czy dwa uruchomienia korzystały z tych samych plików.
Raport JSON zapisujemy atomowo, aby przerwana sesja nie uszkodziła
wcześniejszych wyników.


In [ ]:
# def rozpoczyna definicję funkcji. Jej argumenty path i value dostaną ścieżkę oraz dane do
# zapisania, a wcięte instrukcje wykonają się dopiero po wywołaniu funkcji. Ta funkcja zapisuje
# raport JSON.
def _json_write(path, value):
    """Zapis atomowy chroni poprzedni raport, gdy sesja zostanie przerwana."""
    path = Path(path)
    # Tworzymy katalog nadrzędny. parents=True pozwala utworzyć także brakujące katalogi po
    # drodze, a exist_ok=True pozwala użyć już istniejącego katalogu.
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    # with otwiera plik na czas wykonania wciętego bloku i zamyka go po jego zakończeniu. Najpierw
    # zapisujemy plik tymczasowy, żeby przerwane obliczenia nie uszkodziły poprzedniego raportu.
    with temporary.open("w", encoding="utf-8") as handle:
        # Zamieniamy słowniki i listy Pythona na tekst JSON. ensure_ascii=False zachowuje polskie
        # znaki, a indent=2 ułatwia czytanie pliku.
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)
    # Dopiero po zakończeniu zapisu podmieniamy właściwy plik raportu.
    temporary.replace(path)

# Obliczamy sumę kontrolną zawartości pliku. Dwa pliki o tej samej nazwie mogą zawierać inne dane,
# dlatego w raporcie zapisujemy również ten identyfikator.
def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        # Czytamy plik kolejnymi porcjami po 1 MiB. for powtarza wcięte instrukcje dla każdej
        # porcji; pusty ciąg bajtów b'' oznacza koniec pliku. Dzięki temu nawet duży plik nie musi
        # w całości trafić do RAM.
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    # return kończy funkcję i przekazuje jej wynik do miejsca wywołania. Tutaj wynikiem jest
    # tekstowy zapis sumy kontrolnej.
    return digest.hexdigest()

# Pobieramy plik z adresu url i zapisujemy go pod destination. Jeżeli gotowy plik już znajduje się
# na dysku, możemy go ponownie wykorzystać.
def _download(url, destination):
    """Pobieramy porcjami na dysk; cały duży plik nie trafia do pamięci RAM."""
    destination = Path(destination)
    # if sprawdza warunek. Gdy plik istnieje i ma niezerowy rozmiar, zwracamy jego ścieżkę bez
    # kolejnego pobierania.
    if destination.is_file() and destination.stat().st_size:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Nieukończone pobranie otrzyma końcówkę .part. Właściwa nazwa pliku pojawi się dopiero po
    # pobraniu całej zawartości.
    partial = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "Neuronovo-PolEmo2-education/1.0"})
    print(f"Pobieranie: {destination.name}", flush=True)
    # try obejmuje operacje, które mogą się nie udać, na przykład z powodu przerwania połączenia.
    # Blok except poniżej opisuje postępowanie po takim błędzie.
    try:
        with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as handle:
            # Kopiujemy dane z odpowiedzi serwera do pliku porcjami po 1 MiB. To ważne przy dużym
            # pliku wektorów fastText.
            shutil.copyfileobj(response, handle, length=1024 * 1024)
        partial.replace(destination)
    except Exception:
        # Po nieudanym pobraniu usuwamy niedokończony plik. Instrukcja raise przekaże błąd dalej,
        # żeby raport mógł go wyraźnie pokazać.
        partial.unlink(missing_ok=True)
        raise
    return destination


## Co porównujemy

| Doświadczenie | Zbiór uczący | Walidacja | Test |
|---|---:|---:|---:|
| `all_text`: wszystkie domeny | 6573 teksty | 823 teksty | 820 tekstów |
| `hotels_to_medicine`: hotele → medycyna | 3165 opinii o hotelach | 396 opinii o hotelach | 327 tekstów medycznych |

W drugim doświadczeniu oceniamy te same, już wybrane wagi także na
**395 hotelowych tekstach testowych** (`reference_test`). Możemy wtedy
porównać wynik dla opinii o hotelach z wynikiem dla tekstów medycznych.
Różnica wyników może zależeć również od udziałów klas i trudności tekstów.

Zachowujemy oficjalne podziały autora. Nie tworzymy nowego losowego podziału
i nie łączymy dokumentów `all_text` ze zdaniami `all_sentence`.
Każdy model dopasowujemy osobno w obu doświadczeniach. W doświadczeniu
hotelowym nie korzystamy z wag dostrojonych wcześniej w `all_text`.

Etykiety to `zero` (neutralna), `minus` (negatywna), `plus` (pozytywna)
i `amb` (mieszana/ambiwalentna). Ostatnia klasa obejmuje teksty z elementami
pozytywnymi i negatywnymi; nie muszą one występować w równych proporcjach.


In [ ]:
# Wczytujemy jeden oficjalny podział danych: trening, walidację albo test dla wskazanej domeny.
# Funkcja zwraca tabelę przykładów i informacje o źródle pliku.
def load_split(domain, split, cache_dir="polemo2-cache"):
    """Czytamy format autora: tekst i etykieta w ostatnim tokenie wiersza.

    W danych występują także etykiety plus_m i minus_m. Zgodnie z loaderem
    autora sprowadzamy je do plus i minus. Nie mieszamy wariantu dokumentów
    (text) z wariantem pojedynczych zdań (sentence).
    """
    # Litera f przed napisem pozwala wstawiać wartości zmiennych w miejscach oznaczonych klamrami.
    # Na przykład domena hotels i podział train dadzą nazwę hotels.text.train.txt.
    filename = f"{domain}.text.{split}.txt"
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/data/{filename}"
    path = _download(url, Path(cache_dir) / DATASET_REVISION / filename)
    # Tworzymy dwie puste listy: tekstów i ich prawidłowych etykiet. Kolejne elementy o tym samym
    # indeksie będą dotyczyły tej samej opinii.
    texts, targets = [], []
    # Pętla odczytuje kolejne wiersze pliku. enumerate(..., 1) dodaje numerację od 1, przydatną w
    # komunikacie o ewentualnym błędzie danych.
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        # Autor umieścił etykietę na końcu wiersza. Dzielimy tekst tylko raz, od prawej strony:
        # pierwsza część pozostaje całą opinią, druga zawiera etykietę.
        parts = line.rsplit(maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Brak etykiety: {filename}, wiersz {line_number}.")
        # Rozpakowanie przypisuje dwa elementy listy do dwóch zmiennych w tej samej kolejności.
        text, raw_label = parts
        # Wzorzec rozpoznaje na końcu tokenu jedną z czterech etykiet, również warianty plus_m i
        # minus_m. Dzięki temu zachowujemy mapowanie użyte w oficjalnej funkcji autora wczytującej
        # dane.
        match = re.search(r"(?:^|_)(zero|minus|plus|amb)(?:_m)?$", raw_label)
        if match is None:
            raise ValueError(f"Nieznana etykieta w {filename}, wiersz {line_number}: {raw_label!r}.")
        texts.append(text)
        # append dopisuje element na koniec listy. Nazwę rozpoznanej klasy zamieniamy na liczbę
        # 0–3 zgodnie z kolejnością w LABELS.
        targets.append(LABELS.index(match.group(1)))
    if not texts:
        raise ValueError(f"Pusty zbiór: {filename}.")
    # Z list budujemy tabelę pandas. Kolumna text zawiera opinie, a target prawidłowe numery klas;
    # liczba wierszy jest liczbą przykładów.
    frame = pd.DataFrame({"text": texts, "target": targets})
    metadata = {"filename": filename, "rows": len(frame), "sha256": _sha256_file(path), "source": url}
    # Zwracamy dwie wartości jako krotkę. W miejscu wywołania można je odebrać zapisem dane, opis
    # = load_split(...).
    return frame, metadata


### Zobaczmy prawdziwe przykłady uczące

**DataFrame** z biblioteki pandas jest tabelą. Każdy wiersz odpowiada jednemu tekstowi. Kolumna `text` zawiera tekst, a `target` — numer etykiety. Wczytamy teraz wyłącznie trening z wariantu `all_text`. Podział danych w późniejszym badaniu pozostanie taki jak w źródle.

Funkcja `load_split` zwraca dwa obiekty: tabelę oraz informacje o pliku. Drugą wartość odbierzemy pod nazwą `informacje_o_pliku`. Metoda `head(3)` wybiera trzy pierwsze wiersze do obejrzenia.


In [ ]:
# To jest zbiór uczący. Nie wczytujemy tu testu ani nie wybieramy ustawień.
trening_do_obejrzenia, informacje_o_pliku = load_split(
    "all",
    "train",
)

print("Liczba tekstów uczących:", len(trening_do_obejrzenia))
print("Nazwy kolumn:", trening_do_obejrzenia.columns.tolist())

# Prawidłowe znaczenie numerów pochodzi z opisu zbioru.
numery_etykiet = {0: "zero", 1: "minus", 2: "plus", 3: "amb"}

# copy tworzy osobną tabelę do oglądania, żeby nie zmieniać danych uczących.
podglad_treningu = trening_do_obejrzenia.head(3).copy()

# map zamienia każdy numer w wybranej kolumnie na nazwę ze słownika.
podglad_treningu["nazwa klasy"] = podglad_treningu["target"].map(numery_etykiet)

# display pokazuje tabelę w czytelnej postaci w notatniku.
from IPython.display import display
display(podglad_treningu)

# value_counts liczy, ile przykładów ma każda etykieta.
# sort_index porządkuje wynik według numeru etykiety.
print("Liczebności klas w treningu:")
print(trening_do_obejrzenia["target"].value_counts().sort_index())


**Pytanie:** czy liczebności klas są takie same? Zastanów się, jaki odsetek odpowiedzi uzyskałby model przewidujący zawsze klasę najczęstszą. Za chwilę porównamy odsetek trafień z F1 makro.


## Jak ocenić klasyfikator

Odsetek poprawnych odpowiedzi (ang. accuracy) obliczamy, dzieląc liczbę trafnych przewidywań przez liczbę wszystkich przykładów. Przy nierównych klasach wysoki odsetek trafień może współistnieć ze słabym rozpoznawaniem rzadkich kategorii.

Dla jednej wybranej klasy, np. `minus`, rozróżniamy trzy liczby:

| Skrót | Co liczymy dla klasy `minus` |
|---|---|
| TP | Model przewidział `minus` i prawidłowa etykieta to `minus`. |
| FP | Model przewidział `minus`, choć prawidłowa etykieta jest inna. |
| FN | Prawidłowa etykieta to `minus`, ale model wskazał inną klasę. |

**Precyzja** (ang. precision) odpowiada na pytanie: jaka część wskazań tej klasy była trafna? Liczymy ją jako `TP / (TP + FP)`. **Czułość** (ang. recall) mówi, jaką część rzeczywistych przykładów tej klasy rozpoznaliśmy: `TP / (TP + FN)`.

**F1** łączy te dwie miary: `F1 = 2 × TP / (2 × TP + FP + FN)`. Wysokie F1 wymaga dobrej precyzji i czułości. W tym zadaniu obliczamy F1 osobno dla czterech klas, a ich zwykła średnia to **F1 makro**.

Rozważmy wymyślony zestaw 20 etykiet: 14 negatywnych, 3 pozytywne, 2 neutralne i 1 mieszaną. Model zawsze odpowiada `minus`. Trafi 14 razy, czyli uzyska 70% poprawnych odpowiedzi. Dla `minus` ma TP=14, FP=6 i FN=0, więc F1 wynosi około 82,35%. Dla pozostałych klas F1 wynosi zero. Średnia z czterech wartości to zaledwie około 20,59%.


In [ ]:
# To są wyłącznie wymyślone etykiety do sprawdzenia sposobu liczenia miar.
# Mnożenie listy powtarza jej elementy, a + łączy listy.
prawidlowe_etykiety_pokazowe = (
    ["minus"] * 14
    + ["plus"] * 3
    + ["zero"] * 2
    + ["amb"]
)

# Udajemy klasyfikator, który dla każdego przykładu odpowiada minus.
odpowiedzi_pokazowe = ["minus"] * len(prawidlowe_etykiety_pokazowe)

from sklearn.metrics import accuracy_score, f1_score

trafnosc_pokazowa = accuracy_score(
    prawidlowe_etykiety_pokazowe, odpowiedzi_pokazowe,
)
f1_makro_pokazowe = f1_score(
    prawidlowe_etykiety_pokazowe, odpowiedzi_pokazowe,
    # Jawnie podajemy wszystkie cztery klasy.
    labels=["zero", "minus", "plus", "amb"],
    # macro oznacza zwykłą średnią F1 poszczególnych klas.
    average="macro",
    # Gdy brak przykładów i przewidywań danej klasy, przyjmujemy jej F1 jako zero.
    zero_division=0,
)

# Zapis :.2% wyświetla ułamek jako procent z dwoma miejscami po przecinku.
print(f"Odsetek poprawnych odpowiedzi: {trafnosc_pokazowa:.2%}")
print(f"F1 makro: {f1_makro_pokazowe:.2%}")


**Sprawdź:** skąd bierze się różnica między 70% a 20,59%? Która z miar wyraźniej pokazuje, że trzy klasy nie zostały ani razu rozpoznane? W badaniu używamy F1 makro jako głównej miary i podajemy także pozostałe miary.


## Miary, sprzęt i czas

Główna miara to macro-F1: średnia z F1 policzonego osobno dla każdej
klasy. Każda z czterech klas ma taką samą wagę. Dodatkowo zapisujemy
trafność (`accuracy`) i średnią czułość klas (`balanced_accuracy`).
Wyniki mieszczą się w zakresie od 0 do 1.

W raporcie zapisujemy wersje bibliotek i nazwę sprzętu. Przed pomiarem
czasu czekamy na zakończenie działań GPU, bo są wykonywane asynchronicznie.
Na GPU używamy mieszanej precyzji FP16 (ang. automatic mixed precision),
a na CPU — FP32. Nie obiecujemy konkretnego czasu działania.


In [ ]:
# Zbieramy wersje Pythona i bibliotek. Inna wersja biblioteki może wpływać na działanie kodu lub
# czas obliczeń, więc te informacje należą do opisu doświadczenia.
def _versions():
    result = {"python": platform.python_version()}
    for name in ["numpy", "pandas", "scipy", "scikit-learn", "torch", "transformers", "tokenizers", "safetensors", "huggingface-hub"]:
        try:
            # Zapis słownik[klucz] = wartość dodaje lub zastępuje wpis. Tutaj nazwie biblioteki
            # przypisujemy jej numer wersji.
            result[name] = importlib.metadata.version(name)
        # Brak biblioteki zapisujemy jako None, czyli brak wartości. Sam odczyt metadanych nie
        # wymaga uruchomienia wszystkich metod.
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

# Sprawdzamy, na jakim komputerze wykonano obliczenia. Wynik modelu i czas uczenia to różne
# informacje: sekund z CPU i GPU nie należy porównywać bez znajomości sprzętu.
def _hardware():
    result = {"platform": platform.platform(), "cpu": platform.processor() or platform.machine(), "cpu_count": os.cpu_count()}
    try:
        # W środowisku Colab nazwa procesora znajduje się w systemowym pliku /proc/cpuinfo.
        # Szukamy tylko wiersza z nazwą modelu procesora.
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                result["cpu"] = line.split(":", 1)[1].strip()
                # break kończy najbliższą pętlę. Gdy znaleźliśmy nazwę procesora, nie musimy
                # czytać dalszych wierszy.
                break
    except OSError:
        pass
    try:
        import torch
        result["cuda_available"] = bool(torch.cuda.is_available())
        result["cuda_version"] = torch.version.cuda
        # CUDA pozwala PyTorch wykonywać obliczenia na zgodnym GPU. Dostępność sprawdzamy w
        # bieżącej sesji, ponieważ bezpłatny Colab nie gwarantuje przydzielenia GPU.
        if torch.cuda.is_available():
            # Odczytujemy nazwę pierwszego GPU i jego całkowitą pamięć. Indeks 0 oznacza pierwsze
            # urządzenie.
            prop = torch.cuda.get_device_properties(0)
            result["gpu"] = prop.name
            result["gpu_memory_gib"] = round(prop.total_memory / 1024**3, 2)
    except ImportError:
        result["cuda_available"] = False
    return result

# GPU może nadal liczyć, kiedy Python przechodzi już do następnej instrukcji. Przed odczytem czasu
# czekamy więc na zakończenie zleconych działań.
def _synchronize():
    # Operacje GPU są asynchroniczne. Bez synchronizacji czas byłby zaniżony.
    # Sprawdzamy, czy PyTorch zostało już zaimportowane. Sam pomiar metod klasycznych nie powinien
    # wymagać instalowania tej biblioteki.
    if "torch" in sys.modules:
        import torch
        if torch.cuda.is_available():
            # Ta instrukcja zatrzymuje dalsze wykonywanie kodu do zakończenia bieżących obliczeń
            # CUDA.
            torch.cuda.synchronize()

# Zwracamy odczyt zegara przeznaczonego do pomiaru czasu trwania operacji. Czas działania uzyskamy
# później, odejmując odczyt początkowy od końcowego.
def _clock():
    _synchronize()
    # Odczyt podawany jest w sekundach; może zawierać część ułamkową.
    return time.perf_counter()

# Przygotowujemy ustawienie mieszanej precyzji (ang. automatic mixed precision, AMP). Na GPU
# wybrane operacje użyją liczb FP16, co zmniejsza zużycie pamięci i może przyspieszać obliczenia.
def _amp_context(device):
    """Na GPU używamy mieszanego FP16; na CPU pozostajemy przy FP32."""
    import torch
    # Ten obiekt będzie używany po słowie with. AMP działa tutaj tylko dla CUDA; na CPU obliczenia
    # pozostają w zwykłej precyzji FP32.
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda")

# y_true to prawidłowe klasy, a y_pred odpowiedzi modelu dla tych samych tekstów, w tej samej
# kolejności. Zwracamy trzy miary jakości w słowniku.
def _scores(y_true, y_pred):
    # F1 łączy precyzję, czyli udział poprawnych wskazań danej klasy, z czułością, czyli udziałem
    # odnalezionych przykładów tej klasy. Macro-F1 to średnia z F1 policzonego osobno dla każdej
    # klasy; nie liczymy go z uśrednionej precyzji i czułości. Accuracy oznacza odsetek poprawnych
    # odpowiedzi, a balanced accuracy średnią czułość klas. zero_division=0 ustala wynik równy
    # zero, gdy dla danej klasy nie da się obliczyć miary.
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=range(len(LABELS)), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


## Jak wybieramy parametry i mierzymy wynik

Zbiór **uczący** służy do wyznaczenia wag, **walidacja** do wyboru parametrów,
a **test** do końcowej oceny. Żaden klasyfikator, skaler ani słownik TF-IDF
nie jest dopasowywany do walidacji lub testu.

Główną miarą jest **macro-F1**: liczymy F1 osobno dla każdej z czterech klas,
a potem średnią. Każda klasa ma w tej średniej taką samą wagę. Zapisujemy
również odsetek poprawnych odpowiedzi (`accuracy`) i średnią czułość klas
(`balanced_accuracy`). Wszystkie wartości mieszczą się w zakresie od 0 do 1.

Dla Bayesa sprawdzamy `alpha = 0.1, 1, 10`, a dla regresji logistycznej
`C = 0.1, 1, 4`. Wygrywa najwyższe macro-F1 na walidacji; przy remisie
zostaje pierwszy parametr z tej kolejności. Najlepszą metodę także
wskazujemy według walidacji. Wynik testowy nie służy do podejmowania tych decyzji.

Plan zostaje zapisany w `protocol.json` przed oceną testową. W tym notatniku
nie zmieniaj siatki parametrów po obejrzeniu testu. Własne pomysły sprawdzaj
na treningu i walidacji; powtarzana ocena na tym samym teście przestaje
być niezależnym sprawdzianem.


In [ ]:
# Zapisujemy ustawienia całego doświadczenia przed końcową oceną. Hiperparametry, takie jak C,
# alpha lub liczba epok, wybieramy jako autorzy doświadczenia; zwykłych wag model uczy się z
# danych.
def _configuration():
    """Cały plan zapisujemy przed pierwszym odczytem zbioru testowego."""
    # Słownik zawiera siatki kandydatów i stałe ustawienia uczenia. Walidacja wskaże wariant
    # spośród podanych kandydatów; wynik testu nie będzie służył do ich wyboru.
    return {
        "primary_metric": "macro_f1", "seed": SEED,
        "selection": "Najwyższe macro-F1 walidacji; remis rozstrzyga kolejność siatki.",
        "tfidf": {"word_ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True, "max_features": 60000},
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": 60000},
        "naive_bayes_alpha_grid": [0.1, 1.0, 10.0],
        "logistic_C_grid": [0.1, 1.0, 4.0],
        "logistic_max_iter": 2000, "logistic_class_weight": "balanced",
        "encoder": {"model": ENCODER_ID, "revision": ENCODER_REVISION, "max_length": 256, "batch_size": 16, "learning_rate": 2e-5, "max_epochs": 8, "patience": 2, "min_delta": 1e-4, "weight_decay": 0.01, "restore_best": True, "training_precision": "CUDA: AMP FP16 + GradScaler; CPU: FP32", "inference_precision": "CUDA: AMP FP16; CPU: FP32"},
        "fasttext": {"source": FASTTEXT_URL, "vocabulary": "Unia słów z wejściowych tekstów, bez użycia ich etykiet; wyłącznie filtr pamięci gotowego słownika", "oov": "Pominięte; brak znanych słów daje wektor zerowy"},
        "test_usage": "Etykiety testu służą wyłącznie końcowej ocenie, po wyborze parametrów danej metody. Tekst testowy może wcześniej służyć tylko do wyszukania stałego, gotowego wektora fastText; nie uczymy na nim żadnych wag ani statystyk.",
    }


## Najczęstsza klasa i reprezentacje TF-IDF

Model odniesienia zawsze odpowiada najczęstszą klasą z treningu;
nie czyta tekstu. Kolejne metody zamieniają tekst na TF-IDF słów
i par kolejnych słów. Para „nie polecam” może wnosić inną informację
niż samo „polecam”. Słownik i wagi IDF wyznaczamy tylko na treningu.

Trzecia reprezentacja łączy słowa z fragmentami o długości 3–5 znaków
w obrębie słowa (`char_wb`). Mogą one pomagać przy odmianie wyrazów
i literówkach. Każda część ma limit 60 000 cech.
Regresja logistyczna jest tutaj klasyfikatorem czterech kategorii.


In [ ]:
# Tworzymy opis sposobu zamiany tekstu na cechy TF-IDF. W tym miejscu nie odczytujemy jeszcze
# danych i nie wyznaczamy słownika.
def _word_tfidf():
    # Nie usuwamy częstych słów ani nie sprowadzamy słów do form podstawowych.
    # Słownik oraz wagi IDF powstają wyłącznie na zbiorze uczącym.
    # ngram_range=(1, 2) uwzględnia pojedyncze słowa i pary kolejnych słów. min_df=2 pomija
    # elementy obecne w mniej niż dwóch dokumentach treningowych. sublinear_tf=True zastępuje
    # częstość dodatnią wartością 1 + log(częstości), ograniczając wpływ powtórzeń. Limit słownika
    # wynosi 60 000 cech, a float32 zapisuje liczby w 32 bitach.
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=60000, dtype=np.float32)

# Na podstawie identyfikatora metody wybieramy reprezentację: same słowa albo połączenie słów i
# fragmentów znakowych.
def _features_for(method):
    if method == "tfidf_word_char_lr":
        # FeatureUnion zestawia obok siebie kolumny obu reprezentacji. char_wb tworzy fragmenty o
        # długości 3–5 znaków wewnątrz słów, z uwzględnieniem ich granic. Każda część ma własny
        # słownik i limit 60 000 cech.
        return FeatureUnion([
            ("words", _word_tfidf()),
            ("characters", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=60000, dtype=np.float32)),
        ])
    # Dla pozostałych metod tej grupy przygotowujemy tylko reprezentację słów i par słów.
    return _word_tfidf()

# Przygotowujemy macierze cech treningu i walidacji. W takiej macierzy wiersz odpowiada tekstowi,
# a kolumna jednej cesze, na przykład słowu lub parze słów.
def _prepare_tfidf(context, row, method):
    """Dopasuj słownik i IDF do treningu; walidację tylko przekształć."""
    # Przypisujemy do zmiennej obiekt odpowiedzialny za wektoryzację. W Pythonie obiekt może
    # przechowywać zarówno ustawienia, jak i wyznaczony później słownik.
    vectorizer = _features_for(method)
    started = _clock()
    # fit_transform wykonuje dwa kroki: wyznacza słownik i wagi IDF na treningu, a potem przelicza
    # trening na liczby. Wynik jest macierzą rzadką (ang. sparse): przechowuje głównie wartości
    # niezerowe, co oszczędza pamięć, gdy większość słów nie występuje w danym tekście.
    x_train = vectorizer.fit_transform(context.train_texts)
    row['feature_time'] = _clock() - started
    started = _clock()
    # transform korzysta z już ustalonego słownika i wag IDF. Walidacja nie zmienia reprezentacji;
    # słowa spoza słownika treningowego są pomijane. Trening i walidacja mają te same kolumny
    # cech.
    x_validation = vectorizer.transform(context.validation_texts)
    validation_feature_s = _clock() - started
    # Zapisujemy samą funkcję przekształcania, bez jej wywołania: nie ma tu nawiasów na końcu.
    # Dzięki temu ten sam wyuczony sposób przeliczenia będzie można później zastosować do testu.
    transform = vectorizer.transform
    # Zwracamy słownik z macierzami, czasem przetwarzania i funkcją transform. Nazwane pola
    # ułatwią skorzystanie z tych wyników w dalszych krokach.
    return dict(x_train=x_train, x_validation=x_validation, validation_feature_s=validation_feature_s, transform=transform)

# Ten klasyfikator nie rozpoznaje słów: odpowiada najczęstszą klasą znalezioną w treningu.
# Pokazuje, ile daje sama znajomość częstości klas.
def _run_dummy(context, row):
    """Ucz punkt odniesienia bez czytania tekstu."""
    # estimator to nazwa zmiennej przechowującej model. strategy='most_frequent' wybiera regułę
    # najczęstszej klasy.
    estimator = DummyClassifier(strategy='most_frequent')
    started = _clock()
    # fit uczy model na prawidłowych etykietach treningowych. Macierz zer ma jeden wiersz na tekst
    # i jedną sztuczną cechę; ten model i tak nie wykorzystuje jej wartości. context.train.target
    # odczytuje kolumnę target z tabeli treningowej.
    estimator.fit(np.zeros((len(context.train), 1)), context.train.target)
    row['fit_s'] = row['train_s'] = _clock() - started
    started = _clock()
    # predict zwraca po jednej przewidywanej klasie dla każdego wiersza. Tutaj wszystkie
    # odpowiedzi będą jednakowe.
    predicted = estimator.predict(np.zeros((len(context.validation), 1)))
    validation_s = _clock() - started
    row['validation'] = _scores(context.validation.target, predicted)
    row['selected_params'] = {'strategy': 'most_frequent'}
    # Reguła odpowiedzi została już ustalona z treningu. Teraz możemy sprawdzić ją na odłożonych
    # danych testowych.
    current_test = _get_test(context)
    started = _clock()
    predicted = estimator.predict(np.zeros((len(current_test), 1)))
    test_s = _clock() - started
    _evaluate_reference(context, row, lambda texts: estimator.predict(np.zeros((len(texts), 1))))
    # Zwracamy krotkę czterech wyników. Kolejność jest istotna, bo funkcja wywołująca rozpakowuje
    # je do czterech zmiennych.
    return (predicted, current_test, validation_s, test_s)


## Wybór klasyfikatora na walidacji

Dla naiwnego klasyfikatora Bayesa sprawdzamy `alpha = 0.1, 1, 10`.
Dla regresji logistycznej sprawdzamy `C = 0.1, 1, 4`; mniejsze `C`
mocniej ogranicza wielkość wag. Wygrywa najwyższe macro-F1 walidacji.
Przy remisie zostaje pierwszy parametr z zapisanej kolejności.
`class_weight="balanced"` zwiększa wagę przykładów z rzadszych klas.

Każdy kandydat uczy się wyłącznie na treningu. Dla gęstych wektorów
także skaler dopasowujemy tylko do treningu. Ta funkcja nie ma dostępu
do zbioru testowego.


In [ ]:
# Tworzymy nowy, jeszcze niedopasowany klasyfikator regresji logistycznej dla wybranej wartości C.
def _linear_model(c):
    # Mniejsze C oznacza silniejszą regularyzację, czyli mocniejsze ograniczanie wielkości wag
    # modelu. class_weight='balanced' zwiększa znaczenie błędów na rzadszych klasach, korzystając
    # z ich częstości w treningu. max_iter=2000 jest limitem kroków algorytmu optymalizującego, a
    # nie liczbą tekstów.
    return LogisticRegression(C=c, max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=SEED)

# Sprawdzamy kilka ustawień klasyfikatora. x zawiera cechy liczbowe, y prawidłowe etykiety;
# końcówki train i validation mówią, z którego podziału pochodzą. Funkcja nie otrzymuje testu.
def _select_classifier(method, x_train, y_train, x_validation, y_validation, dense=False):
    """Parametr wybiera macro-F1 walidacji. Przy remisie wygrywa pierwszy.

    Każdy kandydat uczy się od początku na treningu. StandardScaler także
    dopasowujemy wyłącznie na treningu, nigdy na walidacji ani teście.
    """
    # To jawna siatka hiperparametrów. Dodatnia alpha sprawia, że cecha niewystępująca w
    # treningowych tekstach jednej klasy nie otrzyma od razu zerowego oszacowania
    # prawdopodobieństwa dla tej klasy. Większa alpha silniej wygładza oszacowania naiwnego
    # klasyfikatora Bayesa. Parametr C ustala siłę regularyzacji regresji logistycznej. Dla każdej
    # metody sprawdzimy trzy zapisane wartości.
    values = [0.1, 1.0, 10.0] if method == "tfidf_nb" else [0.1, 1.0, 4.0]
    # Rozpakowujemy wartości początkowe do kilku zmiennych. None oznacza, że nie mamy jeszcze
    # zwycięzcy, -1 jest niższe od każdego możliwego F1, a pusta lista history będzie zbierała
    # wyniki kandydatów.
    best, best_score, history, fit_s, validation_s = None, -1.0, [], 0.0, 0.0
    # Pętla wykona cały wcięty blok dla kolejnych wartości parametru. Za każdym razem tworzymy
    # nowy model, żeby kandydaci zaczynali niezależnie.
    for value in values:
        estimator = MultinomialNB(alpha=value) if method == "tfidf_nb" else _linear_model(value)
        # dense oznacza tu gęste wektory fastText lub enkodera. Mają one zapisaną wartość każdej
        # współrzędnej. Dla nich dodajemy skalowanie; macierze TF-IDF pozostawiamy w postaci
        # rzadkiej.
        if dense:
            # Najpierw wykonamy skalowanie przez StandardScaler, a następnie klasyfikację. Podczas
            # fit średnie i odchylenia cech powstaną wyłącznie z x_train. Podczas predict
            # walidacja zostanie przeliczona tymi samymi, już ustalonymi parametrami skalowania.
            # make_pipeline pilnuje tej kolejności kroków.
            estimator = make_pipeline(StandardScaler(), estimator)
        started = _clock()
        # To właściwe uczenie kandydata: model dopasowuje swoje wagi do par cechy–etykieta ze
        # zbioru uczącego.
        estimator.fit(x_train, y_train)
        fit_s += _clock() - started
        started = _clock()
        # Przekazujemy cechy walidacji bez jej etykiet. Prawidłowe odpowiedzi będą potrzebne
        # dopiero przy liczeniu miar, a nie przy tworzeniu przewidywań.
        predicted = estimator.predict(x_validation)
        validation_s += _clock() - started
        metrics = _scores(y_validation, predicted)
        params = {"alpha" if method == "tfidf_nb" else "C": value}
        # Dopisujemy parametry i miary kandydata do historii. Zapis **metrics rozwija wszystkie
        # pary klucz–wartość ze słownika metrics do nowego słownika.
        history.append({"params": params, **metrics})
        print(f"  {params}: walidacja macro-F1={metrics['macro_f1']:.4f}", flush=True)
        # Zmieniamy zwycięzcę tylko po poprawie macro-F1 walidacji. Bardzo małe 1e-12 ogranicza
        # wpływ zaokrągleń liczb zmiennoprzecinkowych; przy remisie zachowujemy pierwszą wartość z
        # siatki.
        if metrics["macro_f1"] > best_score + 1e-12:
            best_score = metrics["macro_f1"]
            # Przechowujemy już dopasowany model zwycięzcy oraz jego wynik i ustawienie. Nie
            # uczymy go ponownie na walidacji.
            best = (estimator, predicted, metrics, params)
    # Gwiazdka rozwija cztery elementy krotki best do wyniku funkcji. Czasy obejmują wszystkich
    # sprawdzonych kandydatów, a nie tylko zwycięzcę.
    return (*best, history, fit_s, validation_s)


## Polskie wektory fastText

Każde znane słowo otrzymuje gotowy wektor 300 liczb, nauczony
wcześniej na polskiej Wikipedii. Tekst opisujemy średnią tych wektorów.
Średnia nie zachowuje kolejności słów.

Oficjalny plik `.vec` czytamy wierszami. Zachowujemy wektory słów
potrzebnych do opisania wejściowych tekstów, także walidacji i testu.
To wyłącznie oszczędność RAM: wektory są stałe, a filtr nie używa
etykiet. Daje te same reprezentacje co cały gotowy słownik.
Format `.vec` nie oblicza wektorów nowych słów z ich fragmentów.
Nieznane słowa pomijamy; tekst bez znanych słów ma wektor zerowy.
Raport podaje pokrycie słownika.


In [ ]:
# Dzielimy tekst na słowa dla metody fastText. To prosty sposób oparty na znakach, odrębny od
# tokenizatora używanego przez model neuronowy.
def _tokens(text):
    # Taka sama prosta tokenizacja w treningu, walidacji i teście.
    # lower zamienia litery na małe, a findall zwraca listę fragmentów pasujących do wzorca.
    # Wzorzec wybiera ciągi znaków słowa, więc na przykład znak kończącej zdanie kropki nie
    # zostanie częścią słowa.
    return re.findall(r"(?u)\b\w+\b", text.lower())

# Wczytujemy potrzebne wektory z gotowego pliku fastText. Nie uczymy tutaj ani wektorów, ani
# klasyfikatora: odczytujemy wartości wyznaczone wcześniej na polskiej Wikipedii.
def _load_fasttext(input_texts, cache_dir):
    """Wczytujemy tylko potrzebną część gotowego, stałego słownika.

    Plik .vec zawiera gotowe wektory słownikowe. Nie jest modelem .bin,
    więc nie potrafi wyliczyć wektora nowego słowa z jego fragmentów.
    Filtrowanie po słowach z wejściowych tekstów oszczędza RAM i daje te same
    wektory co wczytanie całego pliku. Nie używa etykiet ani nie uczy wag.
    Słowa spoza zachowanego słownika pomijamy; pusty tekst ma wektor zerowy.
    """
    # Zapis w klamrach tworzy zbiór (ang. set), czyli kolekcję bez powtórzeń. Dwie pętle w tym
    # zapisie zbierają wszystkie słowa ze wszystkich przekazanych tekstów.
    wanted = {word for text in input_texts for word in _tokens(text)}
    path = _download(FASTTEXT_URL, Path(cache_dir) / "wiki.pl.vec")
    # Pusty słownik będzie przypisywał słowu jego wektor liczb. Wektorem jest tu uporządkowana
    # lista 300 współrzędnych opisujących słowo.
    vectors = {}
    with path.open("rb") as handle:
        # Pierwszy wiersz pliku podaje liczbę słów i długość wektorów. split dzieli ten wiersz na
        # fragmenty oddzielone białymi znakami.
        header = handle.readline().split()
        if len(header) != 2:
            raise ValueError("Niepoprawny nagłówek pliku fastText.")
        dimension = int(header[1])
        # Czytamy kolejne słowa i ich wektory, po jednym wierszu. Dzięki temu pełny wielki słownik
        # nie musi zostać wczytany do RAM.
        for line in handle:
            # Dzielimy wiersz przy pierwszej spacji: przed nią jest słowo, za nią liczby wektora.
            # Odczytujemy bajty z pliku, dlatego następna instrukcja dekoduje słowo do tekstu
            # Unicode.
            raw_word, separator, raw_values = line.rstrip().partition(b" ")
            word = raw_word.decode("utf-8", errors="replace")
            # Liczby przeliczamy tylko dla słów potrzebnych do naszych tekstów. To filtr
            # oszczędzający pamięć; wartości gotowych wektorów pozostają stałe.
            if separator and word in wanted:
                # NumPy zamienia tekst rozdzielonych spacjami liczb na tablicę float32. Sprawdzamy
                # również, czy ma oczekiwaną liczbę współrzędnych.
                vector = np.fromstring(raw_values.decode("ascii"), sep=" ", dtype=np.float32)
                if vector.shape == (dimension,):
                    vectors[word] = vector
    # Zwracamy słownik wektorów, ich wymiar i metadane źródła. Plik .vec nie zawiera mechanizmu
    # wyliczania wektora nowego słowa z jego fragmentów; nieznane słowa będą dalej pomijane.
    return vectors, dimension, {
        "source": FASTTEXT_URL,
        "sha256": _sha256_file(path),
        "dimension": dimension,
        "lookup_vocabulary": len(wanted),
        "retained_vectors": len(vectors),
        "license": "CC-BY-SA-3.0",
    }

# Z wielu wektorów słów tworzymy jeden wektor tekstu: średnią po słowach, dla których mamy gotową
# reprezentację. Taka średnia nie zachowuje kolejności słów.
def _average_vectors(texts, vectors, dimension):
    # Tworzymy gęstą macierz zer: jeden wiersz na tekst i dimension kolumn. Dla fastText dimension
    # wynosi 300. Każdy gotowy wektor tekstu trafi do odpowiedniego wiersza.
    result = np.zeros((len(texts), dimension), dtype=np.float32)
    found = total = empty = 0
    # enumerate zwraca jednocześnie indeks i tekst. Indeks pozwoli wpisać wynik do tego samego
    # wiersza, do którego należy dana opinia.
    for index, text in enumerate(texts):
        words = _tokens(text)
        total += len(words)
        # To lista składana: bierzemy wektor dla każdego wystąpienia znanego słowa. Jeśli słowo
        # powtarza się w tekście, jego wektor również bierze udział w średniej kilka razy.
        available = [vectors[word] for word in words if word in vectors]
        found += len(available)
        if available:
            # axis=0 liczy średnią osobno dla każdej współrzędnej wektorów słów. Wynikiem jest
            # jeden wektor o tej samej długości, a nie jedna liczba dla całego tekstu.
            result[index] = np.mean(available, axis=0)
        else:
            # Jeżeli żadne słowo nie ma wektora w pliku, wiersz pozostaje zerowy. Zliczamy takie
            # teksty, żeby później wiedzieć, jak często wystąpił ten przypadek.
            empty += 1
    # Oprócz macierzy zwracamy pokrycie słownika: udział wystąpień słów, dla których znaleziono
    # wektor, oraz liczbę tekstów z wektorem zerowym.
    return result, {"token_coverage": found / total if total else 0.0, "zero_vectors": empty}

# Przygotowujemy gotowe reprezentacje tekstów, które później otrzyma regresja logistyczna. Uczenie
# klasyfikatora odbędzie się w osobnym kroku.
def _prepare_fasttext(context, row):
    """Wyszukaj stałe wektory; nie dopasowuj ich do etykiet walidacji ani testu."""
    started = _clock()
    # Do wyszukania potrzebnych słów zbieramy same teksty ze wszystkich ocenianych podziałów. Nie
    # korzystamy z etykiet walidacji ani testu. Ponieważ zewnętrzny słownik jest stały, ten filtr
    # daje te same wektory co wczytanie całego pliku; pozwala tylko oszczędzić RAM.
    lookup_texts = context.train_texts + context.validation_texts + _get_test(context).text.tolist()
    reference = _get_reference_test(context)
    if reference is not None:
        lookup_texts += reference.text.tolist()
    # Odbieramy trzy wyniki funkcji: słownik słowo–wektor, długość wektora i metadane źródła.
    vectors, dimension, ft_meta = _load_fasttext(lookup_texts, context.cache_dir)
    row['setup_s'] = _clock() - started
    row['fasttext'] = ft_meta
    started = _clock()
    # Każdy tekst treningowy zamieniamy na średnią wektorów jego słów. Przygotowanie reprezentacji
    # nie wykorzystuje prawidłowej klasy tekstu.
    x_train, training_coverage = _average_vectors(context.train_texts, vectors, dimension)
    row['feature_time'] = _clock() - started
    started = _clock()
    # Walidację przeliczamy za pomocą tych samych gotowych wektorów i tej samej reguły średniej.
    x_validation, validation_coverage = _average_vectors(context.validation_texts, vectors, dimension)
    validation_feature_s = _clock() - started
    row['coverage'] = {'train': training_coverage, 'validation': validation_coverage}
    # lambda tworzy krótką funkcję. Przyjmuje listę tekstów, oblicza ich reprezentacje i wybiera
    # pierwszy wynik [0], czyli samą macierz bez informacji o pokryciu słownika.
    transform = lambda texts: _average_vectors(texts, vectors, dimension)[0]
    row['notes'].append('Stały słownik .vec filtrujemy po słowach z wejściowych tekstów wyłącznie dla oszczędności RAM. Etykiety walidacji i testu nie wpływają na wektory. Brak wektora w oryginalnym pliku oznacza pominięcie słowa.')
    return dict(x_train=x_train, x_validation=x_validation, validation_feature_s=validation_feature_s, transform=transform, vectors=vectors, dimension=dimension)


## Dopasowanie klasyfikatora i końcowa predykcja

Łączymy przygotowaną reprezentację ze wspólną funkcją wyboru parametru.
Zbiór testowy trafia wyłącznie do wybranego modelu. Hotelowy test
odniesienia oceniamy tymi samymi wagami; nie dopasowujemy na nim niczego.
W każdym doświadczeniu przygotowanie reprezentacji i uczenie zaczynają
się osobno, bez przenoszenia dostrojonych wag między doświadczeniami.


In [ ]:
# Ta wspólna funkcja obsługuje modele uczone na gotowej macierzy cech: TF-IDF, średnich fastText
# lub reprezentacjach zamrożonego enkodera.
def _run_vector_method(context, row, method):
    """Wybierz klasyfikator po walidacji; potem oceń wybraną konfigurację."""
    if method == "fasttext_lr":
        # if, elif i else wybierają jedną ścieżkę wykonania. Zależnie od metody przygotujemy inne
        # liczby opisujące te same teksty.
        features = _prepare_fasttext(context, row)
    elif method == "encoder_frozen_lr":
        features = _prepare_frozen(context, row)
    else:
        features = _prepare_tfidf(context, row, method)
    # Zmienna logiczna informuje, czy cechy są gęstymi wektorami. Będzie potrzebna przy decyzji o
    # skalowaniu przed regresją logistyczną.
    dense = method in {"fasttext_lr", "encoder_frozen_lr"}
    # Przekazujemy cechy i etykiety treningu oraz walidacji do wyboru hiperparametru. Otrzymamy
    # już dopasowany najlepszy model; test nie bierze udziału w tym wyborze.
    estimator, _, row["validation"], row["selected_params"], row["validation_history"], row["fit_s"], selection_predict_s = _select_classifier(
        method, features["x_train"], context.train.target,
        features["x_validation"], context.validation.target, dense=dense,
    )
    # Czas uczenia obejmuje przygotowanie cech treningowych i dopasowanie wszystkich kandydatów z
    # siatki. Dzięki temu raport nie pomija kosztu tworzenia reprezentacji ani wyboru C lub alpha.
    row["train_s"] = row["feature_time"] + row["fit_s"]
    validation_s = features["validation_feature_s"] + selection_predict_s
    current_test = _get_test(context)
    started = _clock()
    # Najpierw zamieniamy teksty testowe na cechy za pomocą zapisanej funkcji transform, a potem
    # prosimy wybrany klasyfikator o odpowiedzi. Ten sam wyuczony skaler, jeśli jest potrzebny,
    # działa w połączonych krokach przetwarzania.
    predicted = estimator.predict(features["transform"](current_test.text.tolist()))
    test_s = _clock() - started
    # Funkcja lambda opisuje tę samą pełną drogę tekst–cechy–klasa dla hotelowego testu
    # odniesienia. Nie wykonuje fit.
    _evaluate_reference(context, row, lambda texts: estimator.predict(features["transform"](texts)))
    if method == "fasttext_lr":
        _, coverage = _average_vectors(current_test.text.tolist(), features["vectors"], features["dimension"])
        row["coverage"]["test"] = coverage
    if method == "encoder_frozen_lr":
        row["truncation_rate"]["test"] = _truncation_rate(current_test.text.tolist(), features["tokenizer"])
    return predicted, current_test, validation_s, test_s


## Zapis planu i przygotowanie doświadczenia

Tworzymy katalog wyników, zapisujemy konfigurację i wczytujemy trening
oraz walidację. Przy ponownym uruchomieniu wybranych metod można zachować
inne, wcześniej zapisane wyniki tego samego planu. Każdy wiersz zachowuje
własne informacje o sprzęcie i wersjach bibliotek.


In [ ]:
# Przygotowujemy dane i informacje wspólne dla jednego doświadczenia. context będzie obiektem, z
# którego funkcje metod odczytają trening, walidację, konfigurację i raport.
def _create_context(protocol, output_dir, methods, cache_dir):
    """Zapisz plan i wczytaj wyłącznie trening oraz walidację."""
    'Uruchom jeden protokół; zapisuj raport po zakończeniu każdej metody.\n\n    Parametry methods mogą wskazać np. CLASSICAL_METHODS lub ENCODER_METHODS.\n    Przy ponownym uruchomieniu tego samego protokołu zachowujemy wcześniejsze\n    wiersze raportu, a wskazane metody mierzymy od początku.\n    '
    if protocol not in PROTOCOLS:
        raise ValueError(f'Nieznany protokół: {protocol}. Dostępne: {list(PROTOCOLS)}')
    # Jeżeli methods jest None, wybieramy wszystkie metody. W przeciwnym razie korzystamy z listy
    # podanej przez osobę uruchamiającą notatnik.
    methods = list(METHOD_NAMES) if methods is None else list(methods)
    # Sprawdzamy poprawność listy metod przed rozpoczęciem obliczeń. set usuwa powtórzenia, więc
    # porównanie długości wykrywa powielone identyfikatory; any sprawdza, czy choć jeden
    # identyfikator jest nieznany.
    if not methods or len(set(methods)) != len(methods) or any((method not in METHOD_NAMES for method in methods)):
        raise ValueError('Podaj niepustą listę różnych identyfikatorów z METHOD_NAMES.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Budujemy plan ze wskazanego podziału i wspólnych hiperparametrów. Przejście przez zapis i
    # odczyt JSON ujednolica listy oraz krotki, żeby dało się później porównać plan z zapisanym
    # raportem.
    specification = json.loads(json.dumps({'name': protocol, 'splits': PROTOCOLS[protocol], **_configuration()}))
    # Zapisujemy plan przed końcową oceną testową. Dzięki temu wiadomo, jakie ustawienia
    # przyjęliśmy, zanim poznaliśmy wyniki testu.
    _json_write(output_dir / 'protocol.json', specification)
    data_started = _clock()
    # Gwiazdka rozwija parę z konfiguracji do dwóch argumentów: domeny i nazwy podziału.
    # Otrzymujemy tabelę treningową i opis jej źródła; podobnie wczytujemy walidację.
    train, train_meta = load_split(*PROTOCOLS[protocol]['train'], cache_dir)
    validation, validation_meta = load_split(*PROTOCOLS[protocol]['validation'], cache_dir)
    data_setup_s = _clock() - data_started
    for name, frame in [('train', train), ('validation', validation)]:
        # Sprawdzamy, czy w treningu i walidacji są obecne wszystkie cztery klasy. frame.target
        # odczytuje kolumnę etykiet, a set zostawia ich różne wartości.
        if set(frame.target) != set(range(len(LABELS))):
            raise ValueError(f'Zbiór {name} nie zawiera wszystkich czterech klas.')
    # Rozpoczynamy raport od pustej listy wyników i metadanych: daty, sprzętu, wersji bibliotek
    # oraz źródeł danych. Ten słownik będzie uzupełniany po każdej metodzie.
    report = {'status': 'running', 'run': {'date': datetime.now(timezone.utc).isoformat(), 'protocol': protocol, 'hardware': _hardware(), 'versions': _versions(), 'protocol_config': specification, 'module_sha256': ORIGINAL_MODULE_SHA256, 'data_setup_s': data_setup_s, 'data': {'dataset': DATASET_ID, 'revision': DATASET_REVISION, 'variant': 'all_text' if protocol == 'all_text' else 'hotels_text → medicine_text', 'train_rows': len(train), 'validation_rows': len(validation), 'sources': {'train': train_meta, 'validation': validation_meta}, 'labels': LABELS}, 'notes': ['Wyniki pojedynczego uruchomienia; nie są średnią z wielu losowych inicjalizacji.', 'Teksty, etykiety poszczególnych przykładów i identyfikatory wierszy nie są eksportowane.']}, 'results': []}
    existing_path = output_dir / 'results.json'
    # Jeżeli istnieje wcześniejszy raport, sprawdzamy zgodność planu i wersji danych. Możemy
    # zachować wyniki innych metod, a wskazane obecnie metody policzyć od nowa.
    if existing_path.exists():
        previous = json.loads(existing_path.read_text(encoding='utf-8'))
        if previous.get('run', {}).get('protocol_config') == specification and previous.get('run', {}).get('data', {}).get('revision') == DATASET_REVISION:
            report['results'] = [row for row in previous.get('results', []) if row['id'] not in methods]
    _save_report(report, output_dir)
    # Wyjmujemy kolumny tekstowe i zamieniamy je na listy napisów. Każda pozycja nadal odpowiada
    # temu samemu wierszowi i etykiecie w tabeli.
    train_texts, validation_texts = (train.text.tolist(), validation.text.tolist())
    # Zbieramy wszystko w jednym obiekcie. Początkowe None w polach testu oznacza, że nie został
    # on jeszcze wczytany do tego kontekstu.
    return SimpleNamespace(protocol=protocol, output_dir=output_dir, cache_dir=cache_dir, methods=methods, train=train, validation=validation, train_texts=train_texts, validation_texts=validation_texts, report=report, test=None, test_meta=None, reference_test=None)


## Dwa testy w doświadczeniu ze zmianą domeny

Końcowa ocena korzysta z oficjalnego testu. W doświadczeniu hotelowym
główny test obejmuje 327 tekstów medycznych, a `reference_test` —
395 tekstów o hotelach odłożonych do testu. Żaden nie służy do strojenia.
`domain_gap` oznacza wynik hotelowy minus wynik medyczny. Różnica może
zależeć także od udziałów klas i trudności tekstów.


In [ ]:
# Udostępniamy oficjalny zbiór testowy i dopisujemy jego źródło do raportu. Ta funkcja tylko
# wczytuje dane; nie dopasowuje żadnych wag.
def _get_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    # Wczytujemy plik tylko przy pierwszym użyciu w tym doświadczeniu. Późniejsze metody otrzymają
    # ten sam podział danych.
    if context.test is None:
        context.test, context.test_meta = load_split(*PROTOCOLS[context.protocol]['test'], context.cache_dir)
        context.report['run']['data']['test_rows'] = len(context.test)
        context.report['run']['data']['sources']['test'] = context.test_meta
    # Prawidłowe etykiety z tej tabeli posłużą wyłącznie do obliczenia końcowych miar po wyborze
    # parametrów. Wcześniejszy odczyt samych tekstów dla filtra stałego słownika fastText nie
    # zmienia tego sposobu użycia etykiet.
    return context.test

# Dodatkowy hotelowy test odniesienia ma sens w doświadczeniu, w którym model uczymy na hotelach i
# testujemy na tekstach medycznych.
def _get_reference_test(context):
    """Ocena odłożonych danych bez dopasowywania wag."""
    if context.protocol != 'hotels_to_medicine':
        # Dla doświadczenia all_text kończymy funkcję bez dodatkowego zbioru odniesienia.
        return None
    if context.reference_test is None:
        # Wczytujemy oficjalny hotelowy test, osobny od hotelowego treningu i walidacji. Zawiera
        # 395 tekstów i będzie oceniony tymi samymi wybranymi wagami co test medyczny.
        context.reference_test, metadata = load_split('hotels', 'test', context.cache_dir)
        context.report['run']['data']['reference_test_rows'] = len(context.reference_test)
        context.report['run']['data']['sources']['reference_test'] = metadata
    return context.reference_test

# Otrzymujemy funkcję predict_function zamiast jednego konkretnego rodzaju modelu. Dzięki temu
# możemy tą samą logiką ocenić Bayesa, regresję logistyczną i dostrojony enkoder.
def _evaluate_reference(context, row, predict_function):
    """Ocena odłożonych danych bez dopasowywania wag."""
    reference = _get_reference_test(context)
    if reference is not None:
        started = _clock()
        # Przekazujemy same teksty hotelowego testu odniesienia. Prawidłowe etykiety pojawią się
        # dopiero podczas liczenia miar w następnym kroku.
        predictions = predict_function(reference.text.tolist())
        row['reference_test_predict_s'] = _clock() - started
        # Wynik odniesienia zapisujemy osobno od głównego testu medycznego. Porównanie tych dwóch
        # ocen pokaże różnicę po zmianie domeny, ale nie jest podstawą do dalszego strojenia.
        row['reference_test'] = _scores(reference.target, predictions)


## Co znajdziesz w plikach wynikowych

Raport zapisuje wynik po każdej metodzie, także wtedy, gdy inna metoda
zakończy się błędem. Błąd ma status `error`; nie zastępujemy go liczbą.
Pełne historie walidacji, wybrane parametry i wyniki hotelowego testu
odniesienia są w JSON. CSV zawiera główne miary walidacji i testu.

`train_s` obejmuje przygotowanie reprezentacji treningowych oraz uczenie
wszystkich kandydatów sprawdzanych w siatce, a dla dostrajania — wszystkie
wykonane epoki. Nie jest to wyłącznie czas uczenia zwycięzcy.
`predict_s.validation` obejmuje ocenę kandydatów lub kolejnych epok.
`predict_s.test` mierzy pełną predykcję wybranego modelu, razem z zamianą
tekstów na wektory. Pobranie i wczytanie gotowych wag znajduje się osobno
w `setup_s`. Nie porównuj samych liczb sekund bez sprawdzenia sprzętu.

Raport zawiera wersje bibliotek, nazwę GPU/CPU, rewizje danych i modelu
oraz sumy kontrolne źródeł. Nie zawiera tekstów ani etykiet poszczególnych
przykładów. Pojedyncze uruchomienie nie pokazuje całej zmienności wyniku
wynikającej z losowej inicjalizacji i pracy GPU.


In [ ]:
# Pełny raport zapisujemy jako JSON, a główne miary dodatkowo w tabeli CSV, którą można otworzyć
# na przykład w arkuszu kalkulacyjnym.
def _save_report(report, output_dir):
    output_dir = Path(output_dir)
    _json_write(output_dir / "results.json", report)
    rows = []
    # Każdy element listy results opisuje jedną metodę. Przechodzimy po nich, żeby utworzyć po
    # jednym wierszu tabeli CSV.
    for row in report["results"]:
        # Zapis składany słownika wybiera podstawowe pola wiersza. get zwraca None zamiast
        # zgłaszać błąd, jeśli opcjonalnego pola, na przykład best_epoch, nie ma dla danej metody.
        flat = {key: row.get(key) for key in ["id", "name", "status", "train_s", "setup_s", "feature_time", "fit_s", "best_epoch"]}
        # Zagnieżdżone pętle przechodzą najpierw po walidacji i teście, a potem po ich miarach.
        # Powstaną osobne kolumny, na przykład validation_macro_f1 i test_macro_f1.
        for split in ["validation", "test"]:
            for metric, value in row.get(split, {}).items():
                flat[f"{split}_{metric}"] = value
            flat[f"{split}_predict_s"] = row.get("predict_s", {}).get(split)
        flat["selected_params"] = json.dumps(row.get("selected_params", {}), ensure_ascii=False)
        rows.append(flat)
    if rows:
        # Listę słowników zamieniamy na tabelę i zapisujemy do CSV. index=False pomija dodatkową
        # kolumnę numerów wierszy. Pełne historie uczenia i dodatkowy test odniesienia pozostają w
        # JSON.
        pd.DataFrame(rows).to_csv(output_dir / "results.csv", index=False)

# Przygotowujemy krótszy wydruk raportu, żeby długie historie uczenia nie zajmowały całego ekranu
# Colaba. Pełny raport w pliku pozostaje dostępny.
def _compact_report(report):
    """Wydruk mieści się w Colabie; pełne historie są w pliku JSON."""
    # Tworzymy niezależną kopię również zagnieżdżonych słowników i list. Dzięki temu usunięcie
    # części wydruku nie usunie ich z właściwego raportu.
    compact = copy.deepcopy(report)
    if "protocols" in compact:
        # Funkcja wywołuje samą siebie dla raportu każdego doświadczenia. Takie wywołanie na
        # mniejszej części danych nazywa się rekurencją.
        compact["protocols"] = {key: _compact_report(value) for key, value in compact["protocols"].items()}
    else:
        for row in compact["results"]:
            # pop usuwa wskazane pole z kopii wiersza. Drugi argument None pozwala pominąć pole,
            # jeśli go nie ma.
            row.pop("validation_history", None)
    return compact

# Wykonujemy jedną metodę i zapisujemy jej wynik, czasy oraz ustawienia. Ten krok łączy wcześniej
# opisane funkcje w całe doświadczenie.
def _run_one_method(context, method):
    """Wykonaj metodę, zapisz prawdziwy wynik albo jawny błąd i zwolnij pamięć."""
    print(f'\n[{context.protocol}] {METHOD_NAMES[method]}', flush=True)
    # Tworzymy słownik jednego wiersza raportu. Początkowo status brzmi running; po zakończeniu
    # zmienimy go na completed albo error.
    row = {'id': method, 'name': METHOD_NAMES[method], 'status': 'running', 'setup_s': 0.0, 'feature_time': 0.0, 'fit_s': 0.0, 'truncation_rate': None, 'notes': [], 'selected_params': {}, 'validation_history': []}
    # Każda metoda zachowuje własną datę, sprzęt i wersje bibliotek. Jest to ważne, jeśli część
    # metod uruchomimy w innej sesji niż pozostałe.
    row['run_context'] = {'date': datetime.now(timezone.utc).isoformat(), 'hardware': context.report['run']['hardware'], 'versions': context.report['run']['versions'], 'module_sha256': context.report['run']['module_sha256']}
    try:
        # Wybieramy funkcję wykonującą daną metodę: regułę najczęstszej klasy, klasyfikator na
        # wektorach albo dostrajanie enkodera.
        if method == 'dummy':
            predicted, current_test, validation_s, test_s = _run_dummy(context, row)
        elif method in {'tfidf_nb', 'tfidf_lr', 'tfidf_word_char_lr', 'fasttext_lr', 'encoder_frozen_lr'}:
            predicted, current_test, validation_s, test_s = _run_vector_method(context, row, method)
        else:
            predicted, current_test, validation_s, test_s = _run_finetuned(context, row)
        # Porównujemy końcowe przewidywania wybranego modelu z prawidłowymi klasami testu. Ten
        # wynik trafia do raportu i nie powoduje zmiany hiperparametrów.
        row['test'] = _scores(current_test.target, predicted)
        if 'reference_test' in row:
            # Dla każdej miary odejmujemy wynik medyczny od hotelowego. Dodatnia różnica oznacza
            # wyższy wynik na hotelach. Różnica może zależeć także od udziałów klas i trudności
            # przykładów, a nie wyłącznie od samej tematyki.
            row['domain_gap'] = {key: row['reference_test'][key] - row['test'][key] for key in row['test']}
        row['predict_s'] = {'validation': validation_s, 'test': test_s}
        row['status'] = 'completed'
        print(f"  Wynik: val macro-F1={row['validation']['macro_f1']:.4f}; test macro-F1={row['test']['macro_f1']:.4f}", flush=True)
    # Jeżeli metoda zgłosi błąd, zapisujemy jego typ i treść zamiast zastępować brakujący wynik
    # liczbą. Dzięki temu można sprawdzić problem, a pozostałe metody nadal mogą się wykonać.
    except Exception as error:
        row['status'] = 'error'
        row['error'] = f'{type(error).__name__}: {error}'
        print(f"  Błąd: {row['error']}", flush=True)
    # Dopisujemy gotowy wiersz do listy i od razu zapisujemy raport na dysku. Wyniki zakończonych
    # metod nie muszą czekać na koniec całego porównania.
    context.report['results'].append(row)
    context.report['results'].sort(key=lambda result: list(METHOD_NAMES).index(result['id']))
    _save_report(context.report, context.output_dir)
    # Prosimy Pythona o zwolnienie nieużywanych obiektów. Następnie, jeśli pracujemy na GPU,
    # zwalniamy niewykorzystaną pamięć podręczną PyTorch przed kolejną metodą.
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Wykonanie planu i archiwum

Te funkcje wywołują opisane wyżej metody, łączą ich raporty i tworzą ZIP.
Najlepszą metodę wybierają według walidacji. Jeśli obliczenie zakończy się
błędem, w raporcie zostaje jawny status i komunikat. Nie wpisujemy
zastępczych ani przewidywanych wyników.


In [ ]:
# Uruchamiamy wybrane metody dla jednego planu podziału danych. Domyślne wartości argumentów
# pozwalają wykonać all_text bez podawania wszystkich ustawień ręcznie.
def run_protocol(protocol="all_text", output_dir="polemo2-results", methods=None, cache_dir="polemo2-cache"):
    """Uruchom wybrane metody według jawnego planu jednego doświadczenia."""
    context = _create_context(protocol, output_dir, methods, cache_dir)
    # Kolejno wykonujemy każdą metodę z listy. Wszystkie otrzymają ten sam trening, tę samą
    # walidację i ten sam test w obrębie tego doświadczenia.
    for method in context.methods:
        _run_one_method(context, method)
    # Lista składana wybiera tylko metody zakończone poprawnie. Wierszy z błędem nie używamy do
    # wyboru najlepszego modelu.
    complete_rows = [row for row in context.report['results'] if row['status'] == 'completed']
    if complete_rows:
        # max wybiera wiersz z najwyższym macro-F1 walidacji. Funkcja lambda wskazuje, którą
        # wartość porównujemy. Nie wybieramy zwycięzcy według testu.
        best = max(complete_rows, key=lambda row: row['validation']['macro_f1'])
        context.report['run']['best_method_by_validation'] = best['id']
    context.report['status'] = 'completed' if all((row['status'] == 'completed' for row in context.report['results'])) else 'partial'
    _save_report(context.report, context.output_dir)
    # Zwracamy raport jednego doświadczenia. Można go od razu obejrzeć w Pythonie; kopia została
    # już zapisana w katalogu wyników.
    return context.report

# To główna funkcja uruchamiająca porównanie. Domyślnie wykona wszystkie siedem metod w dwóch
# doświadczeniach, a następnie zapisze wspólny raport i archiwum.
def run_all(output_dir="polemo2-results", methods=None, protocols=None, cache_dir="polemo2-cache"):
    """Uruchom oba doświadczenia i zapisz JSON, CSV oraz archiwum ZIP.

    Przykład krótkiego uruchomienia: run_all(methods=CLASSICAL_METHODS).
    Kolejne run_all(methods=ENCODER_METHODS) dopisze pozostałe metody do tych
    samych raportów, o ile konfiguracja i rewizja danych są takie same.
    """
    # None oznacza wybór wszystkich zdefiniowanych doświadczeń. Można także przekazać własną listę
    # z ich nazwami, na przykład samo all_text.
    protocols = list(PROTOCOLS) if protocols is None else list(protocols)
    if not protocols or any(protocol not in PROTOCOLS for protocol in protocols):
        raise ValueError(f"Dostępne protokoły: {list(PROTOCOLS)}")
    output_dir = Path(output_dir)
    # Rozpoczynamy wspólny raport. W zagnieżdżonym słowniku protocols każda nazwa doświadczenia
    # otrzyma osobny raport siedmiu metod.
    combined = {"status": "running", "protocols": {}}
    for protocol in protocols:
        # Uruchamiamy dane doświadczenie w jego własnym podkatalogu. Oddzielne katalogi
        # rozdzielają raporty, a funkcje treningowe również zaczynają od świeżo wczytanego modelu.
        combined["protocols"][protocol] = run_protocol(protocol, output_dir / protocol, methods=methods, cache_dir=cache_dir)
        combined["status"] = "completed" if all(report["status"] == "completed" for report in combined["protocols"].values()) else "partial"
        _json_write(output_dir / "results.json", combined)
    csv_frames = []
    for protocol in combined["protocols"]:
        csv_path = output_dir / protocol / "results.csv"
        if csv_path.exists():
            frame = pd.read_csv(csv_path)
            # Do wczytanej tabeli wyników dopisujemy kolumnę z nazwą doświadczenia. Dzięki temu po
            # połączeniu tabel wiadomo, z którego podziału pochodzi każdy wiersz.
            frame.insert(0, "protocol", protocol)
            csv_frames.append(frame)
    if csv_frames:
        # Łączymy tabele obu doświadczeń pionowo, czyli dopisujemy ich wiersze jeden pod drugim,
        # zachowując wspólne kolumny miar.
        pd.concat(csv_frames, ignore_index=True).to_csv(output_dir / "results.csv", index=False)
    # Tworzymy archiwum ZIP katalogu wyników. Zawiera raporty JSON i CSV, które można pobrać przed
    # zamknięciem środowiska Colab.
    archive = shutil.make_archive(str(output_dir.resolve()), "zip", root_dir=output_dir)
    print("\nNEURONOVO_RESULTS_JSON", flush=True)
    print(json.dumps(_compact_report(combined), ensure_ascii=False, allow_nan=False), flush=True)
    print(f"Archiwum wyników: {archive}", flush=True)
    # Zwracamy także raport jako zwykły obiekt Pythona. Zapis raport = run_all(...) pozwoli użyć
    # go później do przygotowania tabeli w notatniku.
    return combined


## Uruchomienie porównania

Jedno wykonanie poniższej komórki uruchomi oba doświadczenia: najpierw
`all_text`, a potem hotele → medycyna. W drugim doświadczeniu każdy model
uczy się od początku na danych hotelowych. Porównujemy pięć metod w dwóch doświadczeniach. To 10 uruchomień metod.
Tam, gdzie przewidziano wybór parametrów, wykonujemy również kolejne dopasowania.
Postęp i wyniki walidacji pojawiają się poniżej. Zaczekaj na zakończenie
komórki. Jeśli sesja zostanie przerwana, zachowaj dostępne pliki wyników.


In [ ]:
# Dane i ustawienia obu doświadczeń zapisaliśmy w PROTOCOLS i _configuration.
# To wywołanie uruchamia właściwe uczenie, wybór na walidacji i końcowe testy.
# Każda metoda zapisze swój wynik do raportu; całość otrzyma nazwę raport.
raport = run_all(output_dir="polemo2-results", methods=CLASSICAL_METHODS)


## Zestawienie wyników

Walidacja i test znajdują się w osobnych kolumnach. Dla doświadczenia
hotelowego zobaczysz także wynik testu hotelowego. Najpierw sprawdź,
którą metodę wybralibyśmy według walidacji, a potem odczytaj jej wynik
testowy. Nie zakładamy z góry, która metoda okaże się najlepsza.


### Wyniki opublikowanego uruchomienia

Poniższe liczby pochodzą z zapisanego raportu. Nie są wynikami Twojej bieżącej sesji. Uruchomienie `run_all` tworzy własny raport.

**`all_text`** — data pomiaru: `2026-10-26T20:09:34.693832+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1349 | 0.1462 | 0.4134 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7921 | 0.8119 | 0.8244 |
| TF-IDF słów + regresja logistyczna | 0.8127 | 0.8070 | 0.8171 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8276 | 0.8116 | 0.8159 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7683 | 0.7622 | 0.7646 |
| Zamrożony polski encoder + regresja logistyczna | 0.7813 | 0.7600 | 0.7610 |
| Dostrajanie polskiego encodera | 0.8431 | 0.8366 | 0.8451 |

**`hotels_to_medicine`** — data pomiaru: `2026-10-26T20:19:01.842494+00:00`; GPU: Tesla T4.

| Metoda | Macro-F1 walidacji | Macro-F1 testu | Trafność testu |
|---|---:|---:|---:|
| Najczęstsza klasa | 0.1387 | 0.1446 | 0.4067 |
| TF-IDF słów + naiwny klasyfikator Bayesa | 0.7787 | 0.4865 | 0.6239 |
| TF-IDF słów + regresja logistyczna | 0.8107 | 0.5117 | 0.6636 |
| TF-IDF słów i fragmentów słów + regresja logistyczna | 0.8270 | 0.5585 | 0.6881 |
| Średnia wektorów fastText PL + regresja logistyczna | 0.7965 | 0.5834 | 0.7064 |
| Zamrożony polski encoder + regresja logistyczna | 0.7934 | 0.6012 | 0.7064 |
| Dostrajanie polskiego encodera | 0.8128 | 0.6448 | 0.7248 |

[Pełny raport JSON z metadanymi](https://raw.githubusercontent.com/Neuronovo/notebooks/main/polemo2/results.json).


In [ ]:
import pandas as pd
from IPython.display import display

def pokaz_wyniki(raport):
    # Zestawiamy walidację i test w osobnych kolumnach.
    wiersze = []
    for protokol, czesc in raport["protocols"].items():
        for wynik in czesc["results"]:
            wiersz = {
                "doświadczenie": protokol,
                "metoda": wynik["name"],
                "status": wynik["status"],
                "F1 walidacji": wynik.get("validation", {}).get("macro_f1"),
                "F1 testu": wynik.get("test", {}).get("macro_f1"),
                "trafność testu": wynik.get("test", {}).get("accuracy"),
                "średnia czułość testu": wynik.get("test", {}).get("balanced_accuracy"),
                "uczenie [s]": wynik.get("train_s"),
            }
            if "reference_test" in wynik:
                wiersz["F1 testu hotelowego"] = wynik["reference_test"]["macro_f1"]
            if "error" in wynik:
                wiersz["błąd"] = wynik["error"]
            wiersze.append(wiersz)
    display(pd.DataFrame(wiersze).round(4))


# Własny pomiar ma pierwszeństwo. Bez niego można obejrzeć jawnie
# oznaczony raport opublikowany razem z artykułem.
if "raport" in globals():
    raport_do_podgladu = raport
    print("Wyniki obliczone w tej sesji notatnika.")
else:
    import json
    import urllib.request
    adres_wynikow = "https://raw.githubusercontent.com/Neuronovo/notebooks/main/polemo2/results.json"
    with urllib.request.urlopen(adres_wynikow) as odpowiedz:
        raport_do_podgladu = json.load(odpowiedz)
    print("Podgląd opublikowanego uruchomienia. Nie są to wyniki obliczone w tej sesji.")
pokaz_wyniki(raport_do_podgladu)


## Ćwiczenie: które klasy myli model

Sama średnia nie pokazuje wszystkich błędów. Obejrzymy macierz
pomyłek i kilka błędnie sklasyfikowanych tekstów **z walidacji**.
Odtwarzamy TF-IDF i regresję logistyczną dla `all_text`, używając `C`
wybranego już w raporcie na podstawie walidacji. Model ponownie
dopasowujemy wyłącznie do treningu.

To osobne ćwiczenie. Nie odczytuje testu, nie zmienia raportu
porównania i nie służy do wybierania parametrów po wyniku testowym.
W macierzy wiersze oznaczają klasy prawidłowe, a kolumny przewidywane.
Liczby na przekątnej to poprawne odpowiedzi.


In [ ]:
if "raport" not in globals():
    raise RuntimeError("Najpierw uruchom własne porównanie; podgląd opublikowanych wyników go nie zastępuje.")

import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

wynik_lr = next((w for w in raport["protocols"]["all_text"]["results"]
                if w["id"] == "tfidf_lr" and w["status"] == "completed"), None)
if wynik_lr is None:
    raise RuntimeError("Najpierw uruchom porównanie i sprawdź, czy TF-IDF + LR zakończył się poprawnie.")
wybrane_c = wynik_lr["selected_params"]["C"]
trening, _ = load_split("all", "train")
walidacja, _ = load_split("all", "dev")
model_do_analizy = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                    max_features=60000, dtype=np.float32),
    LogisticRegression(C=wybrane_c, max_iter=2000, class_weight="balanced",
                       solver="lbfgs", random_state=42),
)
model_do_analizy.fit(trening.text, trening.target)
przewidywania = model_do_analizy.predict(walidacja.text)
print("C wybrane wcześniej na walidacji:", wybrane_c)


import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    walidacja.target, przewidywania, labels=range(4),
    display_labels=LABELS, cmap="Blues",
)
plt.title("Macierz pomyłek — walidacja all_text, TF-IDF + LR")
plt.show()

# Oglądamy początek pięciu błędnie sklasyfikowanych opinii.
bledy = walidacja.loc[walidacja.target != przewidywania].copy()
bledy["prawidłowa klasa"] = bledy.target.map(dict(enumerate(LABELS)))
bledy["przewidziana klasa"] = [LABELS[i] for i in przewidywania[walidacja.target != przewidywania]]
bledy["tekst"] = bledy.text.str.slice(0, 500)
display(bledy[["tekst", "prawidłowa klasa", "przewidziana klasa"]].head(5).reset_index(drop=True))


## Pobierz wyniki

Archiwum zawiera raport zbiorczy i osobne raporty doświadczeń. Pobierz
je przed odłączeniem środowiska Colab. Raporty nie zawierają tekstów
poszczególnych opinii; przykłady z ćwiczenia pozostają tylko w jego wyniku.


In [ ]:
from google.colab import files

# Archiwum zawiera raport zbiorczy i osobne raporty obu doświadczeń.
# Pobierz je przed odłączeniem maszyny Colab.
files.download("polemo2-results.zip")


## Źródła

- [PolEmo2: oficjalne repozytorium danych](https://huggingface.co/datasets/clarin-pl/polemo2-official).
- [Kocoń, Miłkowski, Zaśko-Zielińska: opis zbioru PolEmo2](https://aclanthology.org/K19-1092/).
- [fastText: wektory Wikipedii, w tym języka polskiego](https://fasttext.cc/docs/en/pretrained-vectors.html) — CC BY-SA 3.0.
- [sdadas/polish-distilroberta: karta modelu i pliki](https://huggingface.co/sdadas/polish-distilroberta) — Apache 2.0.
- [scikit-learn: TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
  [regresja logistyczna](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html),
  [macro-F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).
- [Google Colab: zasady bezpłatnych zasobów](https://research.google.com/colaboratory/faq.html).

Kod używa dokładnie wskazanych wersji zbioru i modelu (ang. revision),
zapisanych w raporcie.
Szczegółowe warunki korzystania z danych znajdziesz w repozytorium PolEmo2.
